In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

TODO:

- The goal is to create a minimum viable product. Focus on finishing basic common rules.

- Try class for `Card` later? `namedtuple` works for now.
- Explore ways to design `Player` and `Round` classes.
- Try `IntEnum` for Poker hand rankings. Currently, we use integer values from 14 to 1 to indicate the poker hand rankings. Using `IntEnum` may make it more readable. Read more about enum: https://docs.python.org/3/library/enum.html
- Try passing `Round` to `Player` when taking action to update `stage_bet`.
- Player should not be able to check if `stage_bet != player.bet`.
- Add possible actions for players.
- **Odd chips vanish** — `self.pot // len(winners)` drops the remainder
- Implement the rule: If everyone folds except for one player, that player wins
- Try rewriting `evaluate_hand` lazily.
- Implement all-in
- Add pyskills
- Rather than printing, maybe add note messages?
- Add GUI with fasthtml
- Deploy on server

In [ ]:
from fastcore.utils import *
from fastcore.test import *

import random
from collections import namedtuple, Counter
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Callable

# core

> Texas hold'em game logic 

In this dialog, I want to go through the process of writing a poker program myself but following the approach I learned in the cs212 class. My goal is not to copy everything from the lesson exactly, but follow the steps: understanding the problem, specifying, concept inventory, etc. that I can apply to solving any problem. To challenge myself, I will write texas hold'em poker.

Here's the process Norvig taught:

**1. Understand** — Start with a vague understanding and refine it. Make an inventory of all the concepts you'll need to deal with (data types, operations, etc.).

**2. Specify** — Define *what* needs to happen: inputs, outputs, what each function should accomplish. Think of it as a contract — not the implementation details, just the "what."

**3. Design** — Figure out *how* to make it happen — the actual algorithms, code structure, and implementation.

Along the way, key principles:
- **Write tests early** (even before implementation), including **extreme values**
- **Reuse existing tools** when possible
- **Refactor** for elegance (DRY, clarity, simplicity, generality)
- Be aware of the **four dimensions**: correctness, efficiency, features, elegance — and make conscious tradeoffs

## Card

`namedtuple` with `patch`

Let's use emoji for suits and string of alphanumerics for ranks.

In [ ]:
suits = '♠♥♦♣'
suits

'♠♥♦♣'

In [ ]:
ranks = '2,3,4,5,6,7,8,9,10,J,Q,K,A'.split(',')
ranks

['2', '3', '4', '5', '6', '7', '8', '9', '10', 'J', 'Q', 'K', 'A']

Rather than using a string to represent cards and using separate functions to access suits and ranks, I thought it would be better to use the `.` (dot) notation. One way to accomplish this is through `class`, but I wanted to try using `namedtuple`, which allows me to use attribute access and immutability. The ranks and suits from each card should not change. It is more lightweight than `class`.

In [ ]:
Card = namedtuple('Card', ['suit', 'rank'])

c1 = Card('♠', '10')
c1.suit, c1.rank

('♠', '10')

In [ ]:
c1

Card(suit='♠', rank='10')

I wanted the card representation to be more concise, so I used fastcore's `patch` for `__repr__`.

In [ ]:
@patch
def __repr__(self: Card): return self.suit + self.rank

In [ ]:
c2 = Card('♠', 'K')
c2

♠K

Currently, we cannot compare the ranks directly as we are using string comparisons lexicographically. 

In [ ]:
c1.rank > c2.rank, '10' > 'K'

(False, False)

It is convenient to sort hands by rank values when evaluating hands. 

Using `rank_values` mapping, we can convert the string ranks into integers. From 2 to 10, their values are 2 to 10 respectively. Then, from 'J' to 'A', 11 to 14.

In [ ]:
rank_values = {r:i for i,r in enumerate(ranks, start=2)}
rank_values

{'2': 2,
 '3': 3,
 '4': 4,
 '5': 5,
 '6': 6,
 '7': 7,
 '8': 8,
 '9': 9,
 '10': 10,
 'J': 11,
 'Q': 12,
 'K': 13,
 'A': 14}

Although I had `sort_cards` function that sorts cards by their `rank_values` from the highest, it was more convenient to patch `__lt__` and `__eq__` and just use default `sorted` function. When comparing cards, we only care about their ranks, not suits. 

In [ ]:
@patch
def __lt__(self: Card, other: Card): return rank_values[self.rank] < rank_values[other.rank]

In [ ]:
@patch
def __eq__(self: Card, other: Card): return rank_values[self.rank] == rank_values[other.rank]

In [ ]:
deck = [Card(s,r) for s in suits for r in ranks]
sorted(deck,reverse=True)[:5]

[♠A, ♥A, ♦A, ♣A, ♠K]

## Hand

- `evaluate_hand(cards) -> HandRank` — given 7 cards, returns the best 5-card `HandRank`
- `compare_ranks(ranks) -> tuple` — returns the winner rank

A hand has two cards. Community cards have five cards. 

Poker hands:
- royal flush (10)
- straight flush (9, highest_value)
- four of a kind (8, highest_value, kicker)
- full house (7, triple, pair)
- flush (6, five kickers)
- straight (5, highest_value)
- three of a kind (4, triple_value, two kickers)
- two pair (3, high_pair, low_pair, kicker)
- one pair (2, pair_value, three kickers)
- high card (1, highest, second highest, third, fourth, fifth)

### Sample hands

These are some example hands used for examples and tests.

In [ ]:
cards_highcard = [Card(suits[0], ranks[0]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4]),
    Card(suits[3], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[10]), Card(suits[2], ranks[12])]
cards_highcard

[♠2, ♥4, ♦6, ♣8, ♠10, ♥Q, ♦A]

In [ ]:
cards_2kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[3]), Card(suits[1], ranks[5]), 
    Card(suits[2], ranks[7]), Card(suits[3], ranks[9]), Card(suits[0], ranks[11])]
cards_2kind

[♠2, ♥2, ♠5, ♥7, ♦9, ♣J, ♠K]

In [ ]:
cards_3kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[0], ranks[3]), 
    Card(suits[1], ranks[5]), Card(suits[2], ranks[7]), Card(suits[3], ranks[9])]
cards_3kind

[♠2, ♥2, ♦2, ♠5, ♥7, ♦9, ♣J]

In [ ]:
cards_4kind = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]), Card(suits[3], ranks[0]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[7]), Card(suits[2], ranks[10])]
cards_4kind

[♠2, ♥2, ♦2, ♣2, ♠6, ♥9, ♦Q]

In [ ]:
cards_2pair = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[1]), Card(suits[3], ranks[1]),
    Card(suits[0], ranks[4]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_2pair

[♠2, ♥2, ♦3, ♣3, ♠6, ♥8, ♦10]

In [ ]:
cards_fullhouse = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[0], ranks[1]), Card(suits[2], ranks[1]), 
    Card(suits[3], ranks[1]), Card(suits[1], ranks[6]), Card(suits[2], ranks[8])]
cards_fullhouse

[♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10]

In [ ]:
cards_straight = [Card(suits[0], ranks[1]), Card(suits[1], ranks[2]), Card(suits[2], ranks[3]),
    Card(suits[3], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straight

[♠3, ♥4, ♦5, ♣6, ♠7, ♥10, ♦Q]

In [ ]:
cards_flush = [Card(suits[0], ranks[0]), Card(suits[0], ranks[2]), Card(suits[0], ranks[4]),
    Card(suits[0], ranks[6]), Card(suits[0], ranks[8]), Card(suits[1], ranks[3]), Card(suits[2], ranks[7])]
cards_flush

[♠2, ♠4, ♠6, ♠8, ♠10, ♥5, ♦9]

In [ ]:
cards_straightflush = [Card(suits[0], ranks[1]), Card(suits[0], ranks[2]), Card(suits[0], ranks[3]),
    Card(suits[0], ranks[4]), Card(suits[0], ranks[5]), Card(suits[1], ranks[8]), Card(suits[2], ranks[10])]
cards_straightflush

[♠3, ♠4, ♠5, ♠6, ♠7, ♥10, ♦Q]

In [ ]:
cards_royalflush = [Card(suits[0], ranks[12]), Card(suits[0], ranks[11]), Card(suits[0], ranks[10]), 
    Card(suits[0], ranks[9]), Card(suits[0], ranks[8]), Card(suits[1], ranks[2]), Card(suits[2], ranks[4])]
cards_royalflush

[♠A, ♠K, ♠Q, ♠J, ♠10, ♥4, ♦6]

### Ranks and counts

By getting the counts from the cards, we can figure out `four of a kind`, `full house`, `three of a kind`, `two pair`, and `one pair`. One thing we have to be careful of is because we have seven cards to evaluate, `full house` might have two `three of kind`s, rather than a `three of kind` and a `pair`.

Here is `get_ranks_counts`, which takes a list of cards and returns a tuple of ranks and counts. The output is sorted by the rank counts and `rank_values` for tie breakers. Rank counts are useful for finding what kind of poker ranking hand it is, and ranks are used for the main hand components and kickers.

In [ ]:
# limited: Works with `L[arg]`, but loses the parameter info inside L. Use `types.GenericAlias`
@patch
def __class_getitem__(cls: L, *item): return cls

In [ ]:
def get_ranks_counts(cards: list[Card]) -> L[tuple[int, ...], tuple[int, ...]]:
    """Return ((ranks...), (counts...)) sorted by count then by rank value from cards"""
    items = Counter(rank_values[c.rank] for c in cards).items()
    return L(*sorted(items, key=itemgetter(1, 0), reverse=True)).zip()

In [ ]:
get_ranks_counts(cards_fullhouse), cards_fullhouse

([(3, 2, 10, 8), (3, 2, 1, 1)], [♠2, ♥2, ♠3, ♦3, ♣3, ♥8, ♦10])

In [ ]:
def test_get_ranks_counts():
    test_eq(get_ranks_counts(cards_2kind), ((2, 13, 11, 9, 7, 5), (2, 1, 1, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_3kind), ((2, 11, 9, 7, 5), (3, 1, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_4kind), ((2, 12, 9, 6), (4, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_2pair), ((3, 2, 10, 8, 6), (2, 2, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_fullhouse), ((3, 2, 10, 8), (3, 2, 1, 1)))
    test_eq(get_ranks_counts(cards_highcard), ((14, 12, 10, 8, 6, 4, 2), (1, 1, 1, 1, 1, 1, 1)))
    test_eq(get_ranks_counts(cards_straight), ((12, 10, 7, 6, 5, 4, 3), (1, 1, 1, 1, 1, 1, 1)))

    # Edge case: two three-of-a-kinds (e.g. 222 333 K) — full house should pick higher triple
    cards_2trips = [Card(suits[0], ranks[0]), Card(suits[1], ranks[0]), Card(suits[2], ranks[0]),
                    Card(suits[0], ranks[1]), Card(suits[1], ranks[1]), Card(suits[2], ranks[1]),
                    Card(suits[3], ranks[11])]
    test_eq(get_ranks_counts(cards_2trips), ((3, 2, 13), (3, 3, 1)))

test_get_ranks_counts()

### flush

As long as the suit count is 5 or more, it is flush because we have seven cards total.

In [ ]:
def is_flush(cards: list[Card]) -> list:
    "If cards have flush, return the `rank_values` from the cards in sorted order. If not, return []"
    suit, count = Counter(c.suit for c in cards).most_common(1)[0]
    return sorted([c for c in cards if c.suit == suit], reverse=True) if count >= 5 else []

In [ ]:
is_flush(cards_flush)

[♠10, ♠8, ♠6, ♠4, ♠2]

In [ ]:
def test_is_flush():
    # Flush: 5 spades, two side cards
    test_eq([str(c) for c in is_flush(cards_flush)], ['♠10', '♠8', '♠6', '♠4', '♠2'])

    # Straight flush
    test_eq([str(c) for c in is_flush(cards_straightflush)], ['♠7', '♠6', '♠5', '♠4', '♠3'])

    # Royal flush
    test_eq([str(c) for c in is_flush(cards_royalflush)], ['♠A', '♠K', '♠Q', '♠J', '♠10'])

    # No flush: suits too spread out
    test_eq(is_flush(cards_fullhouse), [])
    test_eq(is_flush(cards_highcard), [])
    test_eq(is_flush(cards_4kind), [])
    
    # Edge: all 7 cards same suit — returns all 7
    all_spades = [Card('♠', r) for r in 'A,K,Q,J,10,9,8'.split(',')]
    test_eq([str(c) for c in is_flush(all_spades)], ['♠A', '♠K', '♠Q', '♠J', '♠10', '♠9', '♠8'])
    five_hearts = [Card('♥', r) for r in '2,3,4,5,6'.split(',')] + [Card('♠', 'A'), Card('♦', 'K')]
    test_eq([str(c) for c in is_flush(five_hearts)], ['♥6', '♥5', '♥4', '♥3', '♥2'])

test_is_flush()

### straight

Finding straight is trickier because Ace can serve as 1 or 14. Also, there are seven cards, but only five are used. 

In [ ]:
def is_straight(cards: list[Card]) -> int:
    "Return the highest value of the straight if cards contain 5 consecutive ranks, else 0"
    vals = sorted({rank_values[c.rank] for c in cards}, reverse=True)
    if 14 in vals: vals.append(1)
    for i in range(len(vals) - 4):
        if vals[i] - vals[i+4] == 4: return vals[i]
    return 0

In [ ]:
def test_is_straight():
    # Regular straight: 3-4-5-6-7
    test_eq(is_straight(cards_straight), 7)

    # Ace-high straight (10-J-Q-K-A)
    ace_high = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'), Card('♥','3'), Card('♦','6')]
    test_eq(is_straight(ace_high), 14)

    # Wheel: A-2-3-4-5 (ace low)
    wheel = [Card('♠','A'), Card('♥','2'), Card('♦','3'), Card('♣','4'), Card('♠','5'), Card('♥','9'), Card('♦','J')]
    test_eq(is_straight(wheel), 5)

    # No straight — gapped
    test_eq(is_straight(cards_2pair), 0)

    # No straight — bunch of duplicates that run together in rank
    dupes = [Card('♠','5'), Card('♥','5'), Card('♦','6'), Card('♣','6'), Card('♠','7'), Card('♥','7'), Card('♦','8')]
    test_eq(is_straight(dupes), 0)

test_is_straight()

### evaluate_hand

Now that we can detect flush and straight, we can update `evaluate_hand`.

In [ ]:
def evaluate_hand(cards: list[Card]) -> tuple:
    """Return hand rank tuple for the best 5-card hand from `cards`.
    Hands ranked from 1 (high card) to 10 (royal flush). Each rank tuple
    starts with the hand category, followed by tiebreaker values (kickers
    or relevant ranks)."""
    ranks, counts = get_ranks_counts(cards)
    c0, c1, *rest = counts
    straight_val = is_straight(cards)
    flush_cards = is_flush(cards)
    sf_val = is_straight(flush_cards)
    if sf_val == 14: return (10,) # royal flush
    elif sf_val: return (9, sf_val) # straight flush
    elif c0 == 4: return (8, ranks[0], ranks[1]) # four of a kind
    elif c0 == 3 and c1 >= 2: return (7, ranks[0], ranks[1]) # full house
    elif flush_cards: return (6, *(rank_values[c.rank] for c in flush_cards)) # flush
    elif straight_val: return (5, straight_val) # straight
    elif c0 == 3: return (4, ranks[0], ranks[1], ranks[2]) # three of a kind
    elif c0 == 2 and c1 == 2: return (3, ranks[0], ranks[1], ranks[2]) # two pair
    elif c0 == 2: return (2, ranks[0], ranks[1], ranks[2], ranks[3]) # one pair
    else: return (1, *ranks) # high card

In [ ]:
def test_evaluate_hand():
    # Royal flush: A♠ K♠ Q♠ J♠ 10♠ → (10,)
    test_eq(evaluate_hand(cards_royalflush), (10,))

    # Straight flush: 3♠ 4♠ 5♠ 6♠ 7♠ → (9, 7)
    test_eq(evaluate_hand(cards_straightflush), (9, 7))

    # Four of a kind: 2222 + Q 9 6 → (8, 2, 12)
    test_eq(evaluate_hand(cards_4kind), (8, 2, 12))

    # Full house: 333 22 + 10 8 → (7, 3, 2)
    test_eq(evaluate_hand(cards_fullhouse), (7, 3, 2))

    # Flush: 5 spades → (6, 10, 8, 6, 4, 2)
    test_eq(evaluate_hand(cards_flush), (6, 10, 8, 6, 4, 2))

    # Straight: 3-4-5-6-7 → (5, 7)
    test_eq(evaluate_hand(cards_straight), (5, 7))

    # Three of a kind: 222 + J 9 7 5 → (4, 2, 11, 9)
    test_eq(evaluate_hand(cards_3kind), (4, 2, 11, 9))
    
    # Two pair: 33 22 + 10 8 6 → (3, 3, 2, 10)
    test_eq(evaluate_hand(cards_2pair), (3, 3, 2, 10))

    # One pair: 22 + K J 9 7 5 → (2, 2, 13, 11, 9)
    test_eq(evaluate_hand(cards_2kind), (2, 2, 13, 11, 9))

    # High card: A Q 10 8 6 4 2 → (1, 14, 12, 10, 8, 6, 4, 2)
    test_eq(evaluate_hand(cards_highcard), (1, 14, 12, 10, 8, 6, 4, 2))

    # ace-high straight without flush should NOT be royal flush
    cards_ace_straight_noflush = [Card('♠','10'), Card('♥','J'), Card('♦','Q'), Card('♣','K'), Card('♠','A'), Card('♥','3'), Card('♦','6')]
    test_eq(evaluate_hand(cards_ace_straight_noflush), (5, 14))

    # straight + flush from different cards should NOT be straight flush
    cards_straight_and_flush = [Card('♠','2'), Card('♠','4'), Card('♠','6'), Card('♠','8'), Card('♠','10'), Card('♥','5'), Card('♥','7')]
    test_eq(evaluate_hand(cards_straight_and_flush), (6, 10, 8, 6, 4, 2))

    # ace-high straight + flush, but flush cards not a straight → flush, not royal flush
    cards_ace_straight_nonstraight_flush = [Card('♠','A'), Card('♠','K'), Card('♠','Q'), Card('♠','J'), Card('♠','5'), Card('♥','10'), Card('♥','3')]
    test_eq(evaluate_hand(cards_ace_straight_nonstraight_flush), (6, 14, 13, 12, 11, 5))

    # straight flush hidden when 6+ cards share a suit and top 5 don't form a straight
    cards_sf_hidden = [Card('♠','2'), Card('♠','3'), Card('♠','4'), Card('♠','5'), Card('♠','6'), Card('♠','8'), Card('♠','K')]
    test_eq(evaluate_hand(cards_sf_hidden), (9, 6))

test_evaluate_hand()

### compare_hands

`compare_hands(hands: List[List[Card]]) -> tuple` — Return the winning hand rank among `hands`

In [ ]:
def compare_hands(hands: list[list[Card]]) -> tuple:
    """Return the winning hand rank among `hands`."""
    return max(evaluate_hand(h) for h in hands)

In [ ]:
evaluate_hand(cards_flush)

(6, 10, 8, 6, 4, 2)

In [ ]:
def test_compare_hands():
    # Clear winner: flush beats one pair
    test_eq(compare_hands([cards_2kind, cards_flush]), (6, 10, 8, 6, 4, 2))

    # Tie: two identical hands → both should win
    test_eq(compare_hands([cards_flush, cards_flush]), (6, 10, 8, 6, 4, 2))

    # Three hands, two tie for best
    test_eq(compare_hands([cards_2kind, cards_flush, cards_royalflush]), (10,))
    
    # Reverse order shouldn't matter
    test_eq(compare_hands([cards_flush, cards_2kind]), (6, 10, 8, 6, 4, 2))

test_compare_hands()

### hand_name

In [ ]:
_RANK = {2:'two',3:'three',4:'four',5:'five',6:'six',7:'seven',8:'eight',9:'nine',
         10:'ten',11:'jack',12:'queen',13:'king',14:'ace'}
_HAND = {10:'royal flush',9:'straight flush',8:'four of a kind',7:'full house',
         6:'flush',5:'straight',4:'three of a kind',3:'two pair',2:'one pair',1:'high card'}

In [ ]:
def hand_name(rank: tuple) -> str:
    "Readable name for an `evaluate_hand` tuple, e.g. (7,3,2) -> 'full house, threes over twos'."
    cat, vals = rank[0], rank[1:]
    name = _HAND[cat]
    if cat == 10: return name
    if cat in (9, 5): return f"{name}, {_RANK[vals[0]]}-high"
    if cat in (8, 4): return f"{name}, {_RANK[vals[0]]}s"
    if cat == 7: return f"{name}, {_RANK[vals[0]]}s over {_RANK[vals[1]]}s"
    if cat == 6: return f"{name}, {_RANK[vals[0]]}-high"
    if cat == 3: return f"{name}, {_RANK[vals[0]]}s and {_RANK[vals[1]]}s"
    if cat == 2: return f"{name}, {_RANK[vals[0]]}s"
    return f"{name}, {_RANK[vals[0]]}-high"

In [ ]:
def test_hand_name():
    # Royal flush — only category with no rank mentioned
    test_eq(hand_name((10,)), 'royal flush')

    # Straight flush / straight — "X-high"
    test_eq(hand_name((9, 7)), 'straight flush, seven-high')
    test_eq(hand_name((5, 7)), 'straight, seven-high')
    test_eq(hand_name((5, 14)), 'straight, ace-high')     # ace-high straight
    test_eq(hand_name((5, 5)), 'straight, five-high')     # wheel (A-2-3-4-5)

    # Four of a kind / three of a kind / one pair — "Xs"
    test_eq(hand_name((8, 2, 12)), 'four of a kind, twos')
    test_eq(hand_name((8, 14, 2)), 'four of a kind, aces')
    test_eq(hand_name((4, 2, 11, 9)), 'three of a kind, twos')
    test_eq(hand_name((2, 2, 13, 11, 9)), 'one pair, twos')
    test_eq(hand_name((2, 14, 13, 11, 9)), 'one pair, aces')

    # Full house — "Xs over Ys"
    test_eq(hand_name((7, 3, 2)), 'full house, threes over twos')
    test_eq(hand_name((7, 14, 13)), 'full house, aces over kings')

    # Two pair — "Xs and Ys"
    test_eq(hand_name((3, 3, 2, 10)), 'two pair, threes and twos')
    test_eq(hand_name((3, 14, 13, 12)), 'two pair, aces and kings')

    # Flush / high card — "X-high"
    test_eq(hand_name((6, 10, 8, 6, 4, 2)), 'flush, ten-high')
    test_eq(hand_name((6, 14, 13, 12, 11, 5)), 'flush, ace-high')
    test_eq(hand_name((1, 14, 12, 10, 8, 6, 4, 2)), 'high card, ace-high')
    test_eq(hand_name((1, 2, 4, 6, 8, 10, 12)), 'high card, two-high')

test_hand_name()

## Deck

In my original plan, I thought of writing a `CardCollection` class as an abstract class and create `Hand`, `Deck`, and `CommunityCards` by inheriting from it as they all consist of cards. However, they don't really have methods that share. To keep things simple, I decided to start implementing `Deck` as a list of `Card`s and see how it feels.

A deck consists of 52 cards. From that, we need to shuffle and withdraw cards so the dealer can deal cards.

In [ ]:
def mk_deck(suits: list = suits, ranks: list = ranks) -> L:
    "Create a deck with given suits and ranks"
    return L(Card(s,r) for s in suits for r in ranks)

In [ ]:
def shuffle_deck(deck: list[Card]) -> list[Card]:
    "Shuffle `deck` in place and return it"
    random.shuffle(deck)
    return deck

In [ ]:
random.seed(42)

deck = mk_deck()
shuffle_deck(deck)[:5]

[♠J, ♥Q, ♥A, ♠5, ♥10]

In [ ]:
def test_deck():
    # Standard deck: 52 cards, 4 suits × 13 ranks
    d = mk_deck()
    test_eq(len(d), 52)

    # No duplicates
    test_eq(len(set(d)), 52)

    # Custom: single suit, 5 ranks → 5 cards
    d5 = mk_deck(suits[0], ranks[:5])
    test_eq(len(d5), 5)
    test_eq(d5, [Card('♠','2'), Card('♠','3'), Card('♠','4'), Card('♠','5'), Card('♠','6')])

    # Two suits, three ranks → 6 cards
    d6 = mk_deck(suits[:2], ranks[:3])
    test_eq(len(d6), 6)
    
    # First card is first suit + first rank
    test_eq(d6[0], Card('♠','2'))
    
test_deck()

In [ ]:
def withdraw_card(deck: list[Card], n: int=1) -> list[Card]:
    "Withdraw `n` cards from `deck` and return them. Modifies `deck` in place."
    if len(deck) < n: raise ValueError(f"Not enough cards: asked for {n}, only {len(deck)} left")
    return [deck.pop() for _ in range(n)]

In [ ]:
def test_withdraw_card():
    # One card: returns it, deck shrinks by one, card was last in deck
    deck = mk_deck()
    last_card = deck[-1]
    got = withdraw_card(deck)
    test_eq(got, [last_card])
    test_eq(len(deck), 51)
    
    # n cards: returned in reverse deck order (pop from end)
    deck = mk_deck()
    expected = list(deck[-3:])[::-1]
    got = withdraw_card(deck, 3)
    test_eq(got, expected)
    test_eq(len(deck), 49)

    # Withdrawn cards are exactly the ones removed — nothing lost or duplicated
    deck = mk_deck()
    full = set(deck)
    got = withdraw_card(deck, 5)
    test_eq(len(set(got) & set(deck)), 0)
    test_eq(set(got) | set(deck), full)

    # n=0: no cards, deck unchanged
    deck = mk_deck()
    test_eq(withdraw_card(deck, 0), [])
    test_eq(len(deck), 52)

    # Withdraw the whole deck: works, deck now empty
    deck = mk_deck()
    test_eq(len(withdraw_card(deck, 52)), 52)
    test_eq(len(deck), 0)

    # Too many: raises, and the deck is untouched
    deck = mk_deck()
    test_fail(lambda: withdraw_card(deck, 53), contains='Not enough cards')
    test_eq(len(deck), 52)

    # Depleted deck: raises even for n=1
    deck = mk_deck()
    withdraw_card(deck, 52)
    test_fail(lambda: withdraw_card(deck), contains='Not enough cards')

test_withdraw_card()

## Monte Carlo

## equity

In [ ]:
def equity(hand: list[Card], community: list[Card] = (), n_opponents: int = 1,
           n_sims: int = 10_000, seed: int = None) -> dict:
    """Monte Carlo estimate of win/tie/lose fractions for `hand` given `community` vs `n_opponents`. 
    Pass `seed` for reproducible classroom demos."""
    if n_opponents < 1: raise ValueError('n_opponents must be >= 1')
    hand, community = list(hand), list(community)
    known = set(hand) | set(community)
    deck = [c for c in mk_deck() if c not in known]
    rng = random.Random(seed)
    wins = ties = 0
    fill = 5 - len(community)
    for _ in range(n_sims):
        rng.shuffle(deck)
        n_opp = 2*n_opponents
        board = community + deck[n_opp : n_opp+fill]
        mine = evaluate_hand(hand + board)
        best = max(evaluate_hand(deck[2*j:2*j+2] + board) for j in range(n_opponents))
        if mine > best: wins += 1
        elif mine == best: ties += 1
    return {'win': wins/n_sims, 'tie': ties/n_sims, 'lose': 1-(wins+ties)/n_sims}

The `equity` function provides a great way to simulate what would happen if the game goes on. It counts wins, ties, and loses from 10_000 simulations by default. It's fun to play with. If I start with two aces, what is a chance of winning against one opponent?

In [ ]:
equity([Card('♠','A'), Card('♥','A')])

{'win': 0.8571, 'tie': 0.0053, 'lose': 0.13759999999999994}

It is 85%, but think about what would happen with 2 opponents?

In [ ]:
equity([Card('♠','A'), Card('♥','A')], n_opponents=2)

{'win': 0.7305, 'tie': 0.0042, 'lose': 0.2653}

It dropped to 73%. Why would that be?

What would happen if we have three kings in the community card? How would that change the probability of winning if it changes at all?

In [ ]:
equity([Card('♠','A'), Card('♥','A')], [Card('♠','K'), Card('♥','K'), Card('♦','K')])

{'win': 0.9522, 'tie': 0.0053, 'lose': 0.04249999999999998}

## Player

`Player`class:

- balance: int
- bet: int
- playing: bool
- hand: List[Card] methods:
- take_action(round)
- call(round)
- check(round)
- fold(round)
- raise(round, amount).

In [ ]:
class Action(Enum):
    RAISE = auto()
    CALL = auto()
    CHECK = auto()
    FOLD = auto()

### Strategy

`Player` has a `strategy` attribute, which chooses actions.

In [ ]:
def human(): 
    while True:
        act, *amnt = input("Choose your action. Type 'call', 'check', 'fold', 'raise X' with X to raise the bet.").split()
        if (action := getattr(Action, act.upper(), None)): return (action, int(*amnt) if amnt else 0)
        print(f"Invalid action: {act}. Try again.")

In [ ]:
def always_call(): return (Action.CALL, 0)

def always_check(): return (Action.CHECK, 0)

def always_fold(): return (Action.FOLD, 0)

def always_raise(n):
    def _raise(): return (Action.RAISE, n)
    _raise.__name__ = f'always_raise({n})'
    return _raise

### player class

In [ ]:
@dataclass
class Player:
    username: str = 'new_user' # username
    balance: int = 0 # total balance
    bet: int = 0 # active bet this round
    playing: bool = True # playing this round or not
    hand: list = field(default_factory=list) # playing hand
    strategy: Callable = always_call # a strategy the player uses to make actions (human or ai)

    def perform_action(self, stage_bet) -> int: return ActionHandler(self, stage_bet).handle()
    
    def reset_bet(self): self.bet = 0

    def reset_playing(self): self.playing = True
    
    def call(self, need): return self._place_bet(need)

    def check(self): return 0

    def fold(self): self.playing = False; return 0

    def raise_(self, need): return self._place_bet(need)
    
    def _place_bet(self, amt):
        self.balance -= amt
        self.bet += amt
        return amt

In [ ]:
@patch
def __repr__(self: Player):
    return (f"Player(username={self.username!r}, balance={self.balance}, bet={self.bet}, "
            f"playing={self.playing}, hand={self.hand}, strategy={self.strategy.__name__})")

In [ ]:
class ActionHandler:
    def __init__(self, player, stage_bet):
        self.player = player
        self.stage_bet = stage_bet
    
    def handle(self):
        action, amount = self.player.strategy()
        print(f'{self.player.username} intends to {action.name.lower()}, stage_bet: {self.stage_bet}, raise_amount: {amount}')
        return getattr(self, '_'+action.name.lower())(amount)

    def _call(self, amount):
        need = self.stage_bet - self.player.bet
        if self.player.balance < need:
            if self.player.strategy != human: return self.player.fold() # AI folds when not enough balance for now..
            print(f"Cannot call. balance ({self.player.balance}) too low to match the stage_bet ({self.stage_bet}). Current bet: {self.player.bet}")
            return self.handle()
        else:
            return self.player.call(need)

    def _check(self, amount): return self.player.check()

    def _fold(self, amount): return self.player.fold()

    def _raise(self, amount):
        new_stage_bet = self.stage_bet + amount
        need = new_stage_bet - self.player.bet
        if need > self.player.balance:
            if self.player.strategy != human: return self.player.fold() # AI folds when not enough balance for now..
            print(f'Cannot raise by {amount}. Your balance: {self.player.balance}, current bet: {self.player.bet}, stage_bet: {self.stage_bet}')
            return self.handle()
        else:
            return self.player.raise_(need)

In [ ]:
stage_bet = 2
player1 = Player('galopy', balance=20, playing=True, strategy=always_raise(1))
need = player1.perform_action(stage_bet)
need, player1

galopy intends to raise, stage_bet: 2, raise_amount: 1


(3,
 Player(username='galopy', balance=17, bet=3, playing=True, hand=[], strategy=always_raise(1)))

In [ ]:
def test_player():
    # call: normal case
    p = Player('t', balance=100, bet=10)
    test_eq(p.call(20), 20)
    test_eq(p.balance, 80)
    test_eq(p.bet, 30)

    # call: need = 0 (already matched)
    p = Player('t', balance=100, bet=10)
    test_eq(p.call(0), 0)
    test_eq(p.balance, 100)
    test_eq(p.bet, 10)

    # check: no state change
    p = Player('t', balance=100, bet=10)
    test_eq(p.check(), 0)
    test_eq(p.balance, 100)
    test_eq(p.bet, 10)

    # fold: playing → False, no money changes
    p = Player('t', balance=100, bet=10)
    test_eq(p.fold(), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 100)

    # raise_: normal case
    p = Player('t', balance=100, bet=10)
    test_eq(p.raise_(50), 50)
    test_eq(p.balance, 50)
    test_eq(p.bet, 60)

    # call + raise_ share _place_bet: bets accumulate
    p = Player('t', balance=100, bet=10)
    p.call(5); p.raise_(5)
    test_eq(p.balance, 90)
    test_eq(p.bet, 20)
    
    # reset_bet: only bet zeroed, balance untouched
    p = Player('t', balance=100, bet=10)
    p.reset_bet()
    test_eq(p.bet, 0)
    test_eq(p.balance, 100)

test_player()

In [ ]:
def test_action_handler():
    # call: pays the difference between stage_bet and current bet
    p = Player('t', balance=20, bet=0, strategy=always_call)
    test_eq(p.perform_action(2), 2)
    test_eq(p.balance, 18)
    test_eq(p.bet, 2)
    test_eq(p.playing, True)

    # call: partial — player already bet 1, only needs 1 more
    p = Player('t', balance=20, bet=1, strategy=always_call)
    test_eq(p.perform_action(2), 1)
    test_eq(p.balance, 19)
    test_eq(p.bet, 2)

    # check: no state change, returns 0
    p = Player('t', balance=20, bet=0, strategy=always_check)
    test_eq(p.perform_action(2), 0)
    test_eq(p.balance, 20)
    test_eq(p.bet, 0)
    test_eq(p.playing, True)

    # fold: playing → False, no money moves
    p = Player('t', balance=20, bet=0, strategy=always_fold)
    test_eq(p.perform_action(2), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 20)
    test_eq(p.bet, 0)

    # raise: pays new_stage_bet - current bet
    p = Player('t', balance=20, bet=0, strategy=always_raise(5))
    test_eq(p.perform_action(2), 7)  # 2 + 5 = 7
    test_eq(p.balance, 13)
    test_eq(p.bet, 7)

    # AI folds when it can't afford the call
    p = Player('t', balance=1, bet=0, strategy=always_call)
    test_eq(p.perform_action(2), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 1)

    # AI folds when it can't afford the raise
    p = Player('t', balance=5, bet=0, strategy=always_raise(10))
    test_eq(p.perform_action(2), 0)
    test_eq(p.playing, False)
    test_eq(p.balance, 5)

test_action_handler()

t intends to call, stage_bet: 2, raise_amount: 0


t intends to call, stage_bet: 2, raise_amount: 0


t intends to check, stage_bet: 2, raise_amount: 0


t intends to fold, stage_bet: 2, raise_amount: 0


t intends to raise, stage_bet: 2, raise_amount: 5


t intends to call, stage_bet: 2, raise_amount: 0


t intends to raise, stage_bet: 2, raise_amount: 10


## Round

Round Management

Attributes:
- stage (pre-flop, flop, turn, river)
- players
- deck
- community_cards

Methods:

- `betting_round(round) -> None`: Manage one round of player actions (pre-flop, flop, turn, or river). Ends when all active players have called or checked.
- `distribute(pot, winners) -> None`: Distribute the pot to the winner(s).
- `start_round(round) -> None`: Orchestrate the whole hand: deal hole cards → blinds → betting round → flop → betting round → turn → betting round → river → betting round → showdown → distribute.

In [ ]:
class Stage(Enum):
    PREFLOP = auto()
    FLOP = auto()
    TURN = auto()
    RIVER = auto()

Let's go through playing a round of poker game!

In [ ]:
class Round:
    def __init__(self, players = None, deck = None, community = None, stage = Stage.PREFLOP, stage_bet = 0, pot = 0):
        self.players = [] if players is None else players
        self.deck = shuffle_deck(mk_deck()) if deck is None else deck
        self.community = [] if community is None else community
        self.stage = stage
        self.stage_bet = stage_bet
        self.pot = pot
        self._idx = 0
        self._raiser = -1
    
    def state(self):
        "Return a string describing the current game situation."
        if self.stage is None: return "Game is over"
        player = self.players[self._idx]
        return f"{player.username}'s turn | Stage: {self.stage.name} | Hand: {player.hand} | Community: {self.community} | Stage bet: {self.stage_bet} | Your bet: {player.bet} | Balance: {player.balance} | Pot: {self.pot}"

In [ ]:
@patch
def _distribute(self: Round, winners) -> None:
    "Distribute the prize pot to the winners"
    print(f"Distributing $ {self.pot} to {winners}")
    for player in winners: player.balance += self.pot // len(winners)

In [ ]:
def test_distribute():
    # Single winner takes the whole pot
    r = Round(); r.pot = 10
    p = Player(balance=5)
    r._distribute([p])
    test_eq(p.balance, 15)

    # Even split: two winners
    r = Round(); r.pot = 10
    p1, p2 = Player(balance=5), Player(balance=5)
    r._distribute([p1, p2])
    test_eq(p1.balance, 10)
    test_eq(p2.balance, 10)

    # Odd chips: remainder dropped (documents current behavior)
    r = Round(); r.pot = 5
    p1, p2 = Player(balance=0), Player(balance=0)
    r._distribute([p1, p2])
    test_eq(p1.balance, 2)
    test_eq(p2.balance, 2)

test_distribute()

Distributing $ 10 to [Player(username='new_user', balance=5, bet=0, playing=True, hand=[], strategy=always_call)]


Distributing $ 10 to [Player(username='new_user', balance=5, bet=0, playing=True, hand=[], strategy=always_call), Player(username='new_user', balance=5, bet=0, playing=True, hand=[], strategy=always_call)]


Distributing $ 5 to [Player(username='new_user', balance=0, bet=0, playing=True, hand=[], strategy=always_call), Player(username='new_user', balance=0, bet=0, playing=True, hand=[], strategy=always_call)]


In [ ]:
@patch
def _prep_round(self: Round):
    "Prepare for the round by getting the deck, shuffle, deal hole hands."
    self.deck = shuffle_deck(mk_deck())
    for p in self.players: 
        p.reset_playing()
        p.hand = withdraw_card(self.deck, 2)
    self.community = []
    self.pot = 0
    self.stage = Stage.PREFLOP

In [ ]:
def test_prep_round():
    # Fresh round: each player gets 2 cards, deck loses 2 per player
    players = [Player(balance=20) for _ in range(3)]
    r = Round(players=players)
    r._prep_round()
    test_eq([len(p.hand) for p in players], [2, 2, 2])
    test_eq(len(r.deck), 52 - 6)
    test_eq(r.stage, Stage.PREFLOP)
    test_eq(r.community, [])
    test_eq(r.pot, 0)

    # Called again: state resets, fresh hands dealt
    hands_before = [p.hand for p in players]
    r._prep_round()
    test_eq([len(p.hand) for p in players], [2, 2, 2])
    test_eq(len(r.deck), 52 - 6)
    test_ne([p.hand for p in players], hands_before)

test_prep_round()

In [ ]:
@patch
def _reset_betting_round(self: Round):
    "Reset state for a new betting round."
    self._raiser, self.stage_bet, self._idx = -1, 0, 0
    for p in self.players: p.reset_bet()

In [ ]:
def test_reset_betting_round():
    # Dirty state gets fully reset
    players = [Player(balance=20, bet=3) for _ in range(2)]
    r = Round(players=players)
    r._idx, r.stage_bet, r._raiser = 1, 10, 1
    r._reset_betting_round()
    test_eq([p.bet for p in players], [0, 0])
    test_eq(r.stage_bet, 0)
    test_eq(r._idx, 0)
    test_eq(r._raiser, -1)

    # Balances untouched — only bets reset
    test_eq([p.balance for p in players], [20, 20])

test_reset_betting_round()

In [ ]:
@patch
def _deal_community(self: Round):
    "Deal community cards"
    if self.stage == Stage.FLOP: 
        self.community += withdraw_card(self.deck, 3)
    elif self.stage in (Stage.TURN, Stage.RIVER): 
        self.community += withdraw_card(self.deck)
    print(f"\n--- {self.stage.name} --- Community: {self.community}")

In [ ]:
def test_deal_community():
    # PREFLOP
    r = Round()
    r._deal_community()
    test_eq(len(r.community), 0)
    test_eq(len(r.deck), 52)

    # FLOP: 3 cards dealt, appended to community
    r.stage = Stage.FLOP
    r._deal_community()
    test_eq(len(r.community), 3)
    test_eq(len(r.deck), 52 - 3)

    # TURN: 1 more card appended
    r.stage = Stage.TURN
    r._deal_community()
    test_eq(len(r.community), 4)
    test_eq(len(r.deck), 52 - 4)

    # RIVER: same logic as TURN — 1 card
    r.stage = Stage.RIVER
    r._deal_community()
    test_eq(len(r.community), 5)
    test_eq(len(r.deck), 52 - 5)

test_deal_community()


--- PREFLOP --- Community: []



--- FLOP --- Community: [♣7, ♠J, ♦10]



--- TURN --- Community: [♣7, ♠J, ♦10, ♠4]



--- RIVER --- Community: [♣7, ♠J, ♦10, ♠4, ♥Q]


In [ ]:
@patch
def _get_next_player(self: Round) -> Player:
    while True:
        player = self.players[self._idx]
        if player.playing:
            if player.strategy is human:
                bet_amt = yield player
            else:
                bet_amt = player.perform_action(self.stage_bet)
            self.pot += bet_amt
            if player.bet > self.stage_bet:
                self._raiser = self._idx
                self.stage_bet = player.bet
        self._idx = (self._idx + 1) % len(self.players)
        if self._raiser == -1: self._raiser = 0
        if self._idx == self._raiser: break

In [ ]:
@patch
def _showdown(self: Round):
    "Evaluate and distribute."
    players = L.filter(attrgetter('playing'))(self.players)
    winning_score = compare_hands([p.hand + self.community for p in players])
    winners = [p for p in players if evaluate_hand(p.hand + self.community) == winning_score]
    print(f"Winners: {[p.username for p in winners]} with {winning_score}")
    self._distribute(winners)

In [ ]:
def test_showdown():
    # Clear winner: player1 flush; player2 high card; player3 folded with
    # a winning hand, excluded from evaluation
    community = [Card('♠','6'), Card('♠','8'), Card('♠','10'), Card('♥','5'), Card('♦','9')]
    p1 = Player(username='a', hand=[Card('♠','2'), Card('♠','4')])
    p2 = Player(username='b', hand=[Card('♥','3'), Card('♦','7')])
    p3 = Player(username='c', hand=[Card('♠','A'), Card('♠','K')], playing=False)
    r = Round(players=[p1,p2,p3], community=community, pot=10)
    r._showdown()
    test_eq(p1.balance, 10)
    test_eq(p2.balance, 0)
    test_eq(p3.balance, 0)  # folded: excluded even though hand would win

    # Tie: community royal flush — both players split the pot
    community = [Card('♠','10'), Card('♠','J'), Card('♠','Q'), Card('♠','K'), Card('♠','A')]
    p1 = Player(username='a', hand=[Card('♥','2'), Card('♦','3')])
    p2 = Player(username='b', hand=[Card('♣','4'), Card('♠','5')])
    r = Round(players=[p1,p2], community=community, pot=10)
    r._showdown()
    test_eq(p1.balance, 5)
    test_eq(p2.balance, 5)

test_showdown()

Winners: ['a'] with (6, 10, 8, 6, 4, 2)


Distributing $ 10 to [Player(username='a', balance=0, bet=0, playing=True, hand=[♠2, ♠4], strategy=always_call)]


Winners: ['a', 'b'] with (10,)


Distributing $ 10 to [Player(username='a', balance=0, bet=0, playing=True, hand=[♥2, ♦3], strategy=always_call), Player(username='b', balance=0, bet=0, playing=True, hand=[♣4, ♠5], strategy=always_call)]


In [ ]:
@patch
def start_round(self: Round):
    "Deal, start PREFLOP, and auto-advance to first human player."
    self._prep_round()
    self._reset_betting_round()
    for s in Stage:
        self.stage = s
        self._deal_community()
        self._reset_betting_round()
        yield from self._get_next_player()
    self.stage = None
    self._showdown()

In [ ]:
def test_start_round():
    # A raises 1 each betting round, B calls, C always folds
    a = Player('a', balance=20, strategy=always_raise(1))
    b = Player('b', balance=20, strategy=always_call)
    c = Player('c', balance=20, strategy=always_fold)
    r = Round(players=[a, b, c])
    list(r.start_round())  # consume the generator; all AI, so no yields

    # 4 betting rounds × (A:1 + B:1) = 8 chips, all paid out
    test_eq(sum(p.balance for p in r.players), 60)  # no chips created or lost
    test_eq(a.balance + b.balance, 40)              # 8 chips moved A+B → winner
    test_eq(c.balance, 20)                          # folded, never paid

test_start_round()


--- PREFLOP --- Community: []


a intends to raise, stage_bet: 0, raise_amount: 1


b intends to call, stage_bet: 1, raise_amount: 0


c intends to fold, stage_bet: 1, raise_amount: 0



--- FLOP --- Community: [♥3, ♥K, ♣9]


...


--- RIVER --- Community: [♥3, ♥K, ♣9, ♥2, ♠6]


a intends to raise, stage_bet: 0, raise_amount: 1


b intends to call, stage_bet: 1, raise_amount: 0


Winners: ['a'] with (6, 13, 10, 9, 3, 2)


Distributing $ 8 to [Player(username='a', balance=16, bet=1, playing=True, hand=[♥10, ♥9], strategy=always_raise(1))]


In [ ]:
def act(game, player, round):
    if player is None: print("Game is over"); return
    try:
        player = game.send(player.perform_action(round.stage_bet))
    except StopIteration:
        player = None
    return player

In [ ]:
player1 = Player('galopy', balance=20, playing=True, strategy=always_fold)
player2 = Player('jalopy', balance=20, playing=False)
player3 = Player('zalopy', balance=20, playing=True, strategy=always_call)
players = [player1, player2, player3]
round1 = Round(players = players)
game = round1.start_round()
player = next(game, None)

round1.state()


--- PREFLOP --- Community: []


galopy intends to fold, stage_bet: 0, raise_amount: 0


jalopy intends to call, stage_bet: 0, raise_amount: 0


zalopy intends to call, stage_bet: 0, raise_amount: 0



--- FLOP --- Community: [♠9, ♣A, ♥8]


...

jalopy intends to call, stage_bet: 0, raise_amount: 0


zalopy intends to call, stage_bet: 0, raise_amount: 0


Winners: ['zalopy'] with (4, 8, 14, 13)


Distributing $ 0 to [Player(username='zalopy', balance=20, bet=0, playing=True, hand=[♣J, ♣8], strategy=always_call)]


'Game is over'

In [ ]:
player = act(game, player, round1)
player

Game is over


## Game

We can create a `Game` class that simplifies playing the game.

I want to create a `Game` object that I can use to simplify playing the game. For now, I have to manually initialize players, round, and game. And then iterate the game. How can I design a good `Game` class that works nicely for players?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good time to apply the cs212 lens. Let's start with **Understand**: what's currently awkward?

Looking at your `act` helper and the manual test flow, the friction points are:

1. You build `Player`s by hand (each with a strategy)
2. You create `Round(players=...)` yourself
3. `game = round.start_round()` then `next(game, None)` to kick off
4. Every turn you call `act(game, player, round)` — and you have to carry `round` around, just because `perform_action` needs `stage_bet`
5. You detect "game over" yourself (`player is None`)

So `Game`'s natural job is to hide steps 2–5 — and possibly step 1.

**Specify** — before designing, a few questions that pin down the contract:

1. **Scope: one hand or many?** Does `Game` play a single round, or keep dealing until someone busts / players choose to quit? Your TODO says MVP, so "one hand, then ask to play again" might be enough for now.

2. **Who makes the players?** Should `Game` accept ready-made players, or also handle creating them (buy-in, starting balance)?

3. **What's the playing interface?** Your generator design is already nice — AI turns auto-play, and the game pauses exactly when a human needs to act. Do you want `Game` to expose a `play()` loop, or keep the "yield when it's a human's turn" style you have?

My hunch on #3: keep the yield style, since it already works — `Game` just wraps it so nobody juggles `act()` and `Round` by hand.

How would you answer those three? That'll basically write the specification for you.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 43819, "completion_tokens": 2018, "total_tokens": 45837, "cached_tokens": 37888, "reasoning_tokens": 1632, "cost": 0.002888718}
```


1. one hand, then ask to play again.
2. ready-made players. I want to just do one human vs one bot.
3. keep the yield style.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice, that's a tight spec: one hand, then replay; pre-made players; generator-driven turns. Now for the **Design** stage.

The core question is: **who drives the loop?** Your `act()` helper currently does three jobs — sends the action into the generator, catches `StopIteration`, and carries `round` around. `Game` should absorb all of that. But it should *not* own `input()` — because your TODO mentions a fasthtml GUI later, and the generator+yield design already cleanly separates "game logic" from "UI". A GUI can send actions into the same interface without ever calling `human()`.

So the design shape:

- **State**: `players`, the current `Round`, the live generator, and `current` (the human whose turn it is — `None` means hand over)
- **`start()`** — build `Round`, get generator, advance to first human turn
- **`act()`** — play the current player's turn (strategy handles input), send the bet into the generator, catch `StopIteration` → `current = None`
- **`play()`** — convenience: `start()` then loop `act()` until hand over; the caller wraps this in the "play again?" loop

One detail: `act` needs `stage_bet` for `perform_action`, but that's easy — it lives on the round, which `Game` holds.

A skeleton to react to:

```python
class Game:
    def __init__(self, players):
        self.players = players
        self._round = None
        self._gen = None
        self.current = None  # human player to act; None = hand over

    def start(self):
        self._round = Round(players=self.players)
        self._gen = self._round.start_round()
        self.current = next(self._gen, None)

    def act(self):
        if self.current is None: return
        try:
            self.current = self._gen.send(self.current.perform_action(self._round.stage_bet))
        except StopIteration:
            self.current = None

    def play(self):
        self.start()
        while self.current is not None:
            self.act()

game = Game([Player('you', balance=20, strategy=human),
             Player('bot', balance=20, strategy=always_call)])
game.play()
```

Does this match your mental model? One thing to check: with one human + one bot, `current` will always be the human — but what if you later want multiple humans? The yield style already handles it, since `_get_next_player` only yields for `human` strategy players.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 44296, "completion_tokens": 3239, "total_tokens": 47535, "cached_tokens": 22016, "reasoning_tokens": 2671, "cost": 0.006501516000000001}
```


I don't like how it keeps looping until the game ends. Before the human player gets to take an action with an `input` prompt, he has to be able to use solveit environment to explore options, ask questions to solveit ai, play with code, etc. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Right — a blocking `play()` loop defeats the whole point of solveit. The generator already gives you discrete steps; the fix is to never loop, and expose one-turn-at-a-time as the *primary* interface:

```python
class Game:
    def __init__(self, players):
        self.players = players
        self._round = None
        self._gen = None
        self.current = None  # human to act; None = hand over

    def start(self):
        "Deal a fresh hand and advance to the first human turn."
        self._round = Round(players=self.players)
        self._gen = self._round.start_round()
        self.current = next(self._gen, None)

    def act(self):
        "Play one human turn (AI turns auto-play inside the generator)."
        if self.current is None: return
        try:
            self.current = self._gen.send(self.current.perform_action(self._round.stage_bet))
        except StopIteration:
            self.current = None

    def state(self):
        return self._round.state() if self._round else "Game is over"
```

Usage becomes — deal, then a cell per turn, with anything you like in between:

```python
game = Game([Player('you', balance=20, strategy=human),
             Player('bot', balance=20, strategy=always_call)])
game.start()
game.state()
```

```python
game.act()   # your turn; bot's reply auto-plays
game.state()
```

Two design notes:

1. **`play()` disappears entirely.** "One hand, then ask to play again" now just means: when `state()` says "Game is over", you decide whether to call `game.start()` again — no loop needed.

2. **`act()` already advances past all AI turns**, because `_get_next_player` only yields for `human`-strategy players. So one `act()` call = one human decision, then it's your turn again or the hand is over. The generator does the bookkeeping.

One thing worth checking when you run it: `human()` uses `input()`, which may block or misbehave in a solveit code cell — worth testing early rather than discovering it mid-game. If it doesn't work, an easy alternative is a `human_action(action, amount)` strategy you call from a cell, which fits the "explore then act" flow even better.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 44976, "completion_tokens": 2103, "total_tokens": 47079, "cached_tokens": 38272, "reasoning_tokens": 1588, "cost": 0.0030948120000000006}
```


In [ ]:
class Game:
    def __init__(self, players):
        self.players = players
        self._round = None
        self._gen = None
        self.current = None  # human to act; None = hand over

    def start(self):
        "Deal a fresh hand and advance to the first human turn."
        self._round = Round(players=self.players)
        self._gen = self._round.start_round()
        self.current = next(self._gen, None)
        print(self.state())

    def act(self):
        "Play one human turn (AI turns auto-play inside the generator)."
        if self.current is None: return
        try:
            self.current = self._gen.send(self.current.perform_action(self._round.stage_bet))
        except StopIteration:
            self.current = None
        print(self.state())

    def state(self): return self._round.state() if self._round else "Game is over"

In [ ]:
game = Game([Player('you', balance=20, strategy=human),
             Player('bot', balance=20, strategy=always_call)])
game.start()


--- PREFLOP --- Community: []


you's turn | Stage: PREFLOP | Hand: [♥8, ♣5] | Community: [] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0


In [ ]:
game.act()

you intends to call, stage_bet: 0, raise_amount: 0


bot intends to call, stage_bet: 0, raise_amount: 0



--- FLOP --- Community: [♦6, ♦J, ♦9]


you's turn | Stage: FLOP | Hand: [♥8, ♣5] | Community: [♦6, ♦J, ♦9] | Stage bet: 0 | Your bet: 0 | Balance: 20 | Pot: 0


In [ ]:
game.act()

you intends to raise, stage_bet: 0, raise_amount: 1


bot intends to call, stage_bet: 1, raise_amount: 0



--- TURN --- Community: [♦6, ♦J, ♦9, ♥J]


you's turn | Stage: TURN | Hand: [♥8, ♣5] | Community: [♦6, ♦J, ♦9, ♥J] | Stage bet: 0 | Your bet: 0 | Balance: 19 | Pot: 2


In [ ]:
game.act()

you intends to raise, stage_bet: 0, raise_amount: 2


bot intends to call, stage_bet: 2, raise_amount: 0



--- RIVER --- Community: [♦6, ♦J, ♦9, ♥J, ♦3]


you's turn | Stage: RIVER | Hand: [♥8, ♣5] | Community: [♦6, ♦J, ♦9, ♥J, ♦3] | Stage bet: 0 | Your bet: 0 | Balance: 17 | Pot: 6


In [ ]:
game.act()

you intends to raise, stage_bet: 0, raise_amount: 2


bot intends to call, stage_bet: 2, raise_amount: 0


Winners: ['bot'] with (6, 13, 11, 9, 6, 3)


Distributing $ 10 to [Player(username='bot', balance=15, bet=2, playing=True, hand=[♦K, ♥Q], strategy=always_call)]


Game is over


In [ ]:
game.players

[Player(username='you', balance=15, bet=2, playing=True, hand=[♥8, ♣5], strategy=human),
 Player(username='bot', balance=25, bet=2, playing=True, hand=[♦K, ♥Q], strategy=always_call)]

It works pretty well

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice — and good news on the earlier concern: `input()` worked fine from a code cell, so `human()` strategy is usable as-is for now.

One small thing worth noticing in the output: at the start of each new stage, `stage_bet` and `Your bet` reset to 0 (that's `_reset_betting_round` doing its job), so `Balance` is the only number that carries over. That's correct game logic — just make sure the state display reads naturally to you before you show it to anyone.

Replay is exactly one call away: `game.start()` again deals a fresh hand with the same players.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 46551, "completion_tokens": 633, "total_tokens": 47184, "cached_tokens": 44032, "reasoning_tokens": 497, "cost": 0.0017195220000000002}
```


At this point, I am thinking of creating pyskill for this project. Do you know what pyskill is?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — pyskills is a plugin system where Python packages register "skills" (units of LLM-usable functionality) via standard entry points. An LLM host (like solveit) discovers them without importing anything — it reads lightweight descriptions via AST inspection — and selectively loads chosen ones into context on demand.

The key pieces:

- **`list_pyskills()`** returns `{module: description}` for everything registered; no imports needed
- **Loading** is just `import` + `doc()` — first the module, then each class/function before first use
- Each skill's `__all__` is curated, so `from <module> import *` brings in exactly the intended API
- `doc()` works on any module, not just registered ones — registration only affects discovery

You've actually already used several in this dialog: `fastcore.tools` (file edits), `rgapi.skill` (search), `exhash.skill` (hash-verified editing), and `dialoghelper.solveitskill`.

For your poker project, a pyskill could register things like the game's public API — `Game`, `Round`, `Player`, maybe a "deal a hand" helper — so an LLM in a solveit dialog can discover and drive the game without you having to explain the whole API each time.

What are you thinking of exposing for holdem?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 55342, "completion_tokens": 349, "total_tokens": 55691, "cached_tokens": 4224, "reasoning_tokens": 60, "cost": 0.010970064000000002}
```


Those are good APIs. This project is about building an LLM agent that teaches how to play texas holdem poker games and also about probability. Take a look at `index.ipynb` in this directory where I list concepts and learning materials students can learn while playing the poker.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me look at the index dialog:
- ⏳ `py(code="await view_dlg('index')")` ⏳


````json {.tool}
{
  "id": "call_00_M8ieDLK7YVOnuvhA8ESm9307",
  "name": "py",
  "args": {
    "code": "await view_dlg('index')"
  },
  "result": "<TRUNCATED>…kdown id=\"d1b77a09\">This project is about building an LLM agent that teaches how to play texas holdem poker games and also about probability. </markdown><prompt id=\"62273b20\">What concepts can people learn from playing poker games?<out>Poker is a surprisingly rich playground for learning. Here are the main concept families, in rough order of how they usually come up:\n\n**Probability & combinatorics**\n- Counting outcomes: how many ways to make a flush, a straight, a pair\n- Conditional probability: \"given what I hold and what's on the board, what's my chance of winning?\"\n- Pot odds: comparing the chance of winning to the price of calling\n\n**Decision-making under uncertainty**\n- Expected value: weighing the size of a win against its probability\n- Expected utility: when a bigger win isn't worth the risk — risk tolerance\n- Bayesian reasoning: updating your estimate of an opponent's hand as they act\n\n**Game theory & strategy**\n- Position and information: how acting later lets you see more before deciding\n- Ranges: thinking in sets of hands instead of one specific hand\n- Mixed strategies and Nash equilibrium (the \"GTO\" idea — making yourself unpredictable on purpose)\n\n**Statistics & psychology**\n- Variance: why a good player can lose 10 hands in a row\n- Sample size: judging skill from results only over many hands\n- Emotional regulation: tilt, discipline, not chasing losses\n\nThe probability and expected-value parts map directly onto math you'd learn in a statistics class; the strategy parts map onto game theory, which is its own fascinating branch of math.\n\nWhich of these would you like to dig into first?\n\n```json {.usage}\n{\"model\": \"deepseek-v4-flash\", \"prompt_tokens\": 28048, \"completion_tokens\": 783, \"total_tokens\": 28831, \"cached_tokens\": 4224, \"reasoning_tokens\": 492, \"cost\": 0.005420604000000001}\n```\n</out></prompt><prompt id=\"79faa5d7\">Let's dive into probability and math first. Are there more concepts than those three? Include everything you can think of. Do a web search to get ideas, too.<out>Let me search for a comprehensive list of math concepts in poker.\n- ⏳ `web_answer(pr=\"List all the probability and mathematics…\", qs=\"['math concepts in poker probability lis…\")` ⏳\n\n\n```json {.tool}\n{\n  \"id\": \"call_00_Moj8iqLFMyex9s6OPyu61952\",\n  \"name\": \"web_answer\",\n  \"args\": {\n    \"pr\": \"List all the probability and mathematics concepts that can be learned or applied in Texas holdem poker, as comprehensively as possible. Include topics like combinatorics, pot odds, expected value, variance, Bayesian reasoning, game theory, equity, fold equity, implied odds, and anything else mathematical that comes up in poker strategy.\",\n    \"qs\": [\n      \"math concepts in poker probability list\",\n      \"poker mathematics concepts combinatorics pot odds expected value\",\n      \"Texas holdem math probability concepts learn\",\n      \"poker math topics list statistics game theory\"\n    ]\n  },\n  \"result\": \"<TRUNCATED>…he provided search results, here is a comprehensive list of probability and mathematics concepts applicable to Texas Hold'em poker, organized by category.\\n\\n> **Note regarding \\\"opened pages\\\":** The given opened pages returned errors (invalid URLs and a parsing failure), so I was unable to extract any information from them. All citations below are drawn solely from the provided search result snippets and titles.\\n\\n---\\n\\n## 1. Core Probability & Combinatorics\\n\\n- **Hand Probability & Frequency Counting**: Calculating the likelihood of being dealt specific starting hands and the probability of completing certain hand rankings. The total number of 5-card poker hands is 2,598,960, and the total number of Texas Hold'em starting hand combinations is 133,784,560. These counts form the basis of all probability calculations.¹ ²\\n- **Outs & the Rule of 2 and 4**: Counting the number of cards remaining in the deck that will improve your hand (outs) and estimating the probability of winning on the next street (Rule of 2: outs × 2 = % hit) or by the river (Rule of 4: outs × 4 = % hit).³ ⁴\\n- **Combinatorics (Combination Counting)**: Counting the number of ways a specific hand can be dealt (e.g., 6 combos of a pocket pair, 16 combos of an unpaired hand) and using blocker effects to refine opponent's possible hands. This is critical for estimating the frequency of bluffs versus value hands in an opponent's range.⁵ ⁶\\n\\n## 2. Odds, Ratios, and Pot Math\\n\\n- **Pot Odds**: The ratio of the current pot size to the cost of a contemplated call. Used to determine if a call is mathematically profitable in the long run (e.g., if the pot is $100 and it costs $20 to call, your pot odds are 5:1).⁷ ⁸\\n- **Implied Odds**: An extension of pot odds that accounts for additional money you expect to win from your opponent on future streets if you make your hand.⁵ ⁹\\n- **Fold Equity**: The probability that your opponent will fold to a bet, combined with your hand's equity to determine the expected value of a bet or raise. (Though not e…</TRUNCATED>"
}
````





Read it — the vision is clear: an interactive tutor that teaches poker *rules* and *probability* through doing, with a real deck (full or small), real simulations, and plots. That sharpens what the pyskill needs to be: not just the game API, but a *teaching-oriented* layer on top.

Mapping the tutor's needs to your existing core API:

| Tutor need | Core API already has |
|---|---|
| Explain rules / show a hand flow | `Round`, `Game`, `Player`, `Stage` |
| Deal specific cards for examples | `Card`, `mk_deck`, `withdraw_card` |
| "Small deck" for simple probability | `mk_deck(suits, ranks)` already accepts custom suits/ranks — e.g. `mk_deck(suits[:2], ranks[:5])` = 10 cards |
| Why did X beat Y | `evaluate_hand`, `compare_hands` |
| Simulate many hands (law of large numbers) | Loop `Round` + strategies |
| Students act and see consequences | `Game` with `human` / bot strategies |

**The gaps** — things the tutor needs that core doesn't have yet:

1. **Probability helpers**: e.g. `equity(hand, community, n_opponents, n_sims)` via Monte Carlo — the single most useful number for "what should I do here?" questions.
2. **Readable hand names**: `evaluate_hand` returns `(7, 3, 2)`; the tutor wants "full house, threes over twos" — a `hand_name(rank_tuple)` function.
3. **Deterministic setup**: for examples you often want to force a specific board ("make a board where you have a flush draw") — worth a small helper or just documented idioms.
4. **The docstring itself** — a pyskill's module docstring is the LLM's instruction manual: it should teach the *workflow* (small deck first, then simulate, then show the plot), not just list functions.

Two design questions:

1. **Where does it live?** Pyskills register via entry points, so the skill should ship inside the `holdem` package (e.g. `holdem.skill`), discoverable whenever the package is installed.

2. **Pure vs. teaching layer?** Do you want probability helpers (`equity`, `hand_name`) in core (so the GUI could use them too), or only in the skill — keeping core a clean game-logic library and the skill a curated tutor-facing facade? The pyskill philosophy leans toward the latter.

Which way are you leaning on those?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 115418, "completion_tokens": 2911, "total_tokens": 118329, "cached_tokens": 111232, "reasoning_tokens": 2289, "cost": 0.004437552000000001}
```


1. I agree.
2. Latter.

This project is lean. If students want to explore some areas not covered by the curriculum or tools from the project, they have solveit AI to help them build tools and explore topics.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a clean division of labor: core stays a pure game-logic library, the skill is the tutor-facing facade, and anything beyond that students build themselves with the assistant's help.

Next concrete step is learning how to register a pyskill — there's a `createskill` helper in the pyskills package:

```python
from pyskills import createskill; doc(createskill)
```

That will show you the entry-point mechanics and the minimal module shape. Want to run that yourself and see what it says, or should I pull up the docs here?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 57781, "completion_tokens": 468, "total_tokens": 58249, "cached_tokens": 55808, "reasoning_tokens": 348, "cost": 0.0017828580000000002}
```


In [ ]:
from pyskills import createskill; doc(createskill)

---

### pyskills.createskill

```python
module instance: <module 'pyskills.createskill' from '/usr/local/lib/python3.12/site-packages/py…
```

*How to create a pyskills pyskill module.*

A pyskill is a standard Python module that registers itself via entry points so LLM hosts can discover and load it.

## 1. Create your module

Your module needs:
- A docstring: first paragraph is the short description shown during discovery; the rest is read by the LLM after loading.
- `__all__` (optional): if provided, `doc()` and `xdir()` show exactly these symbols. Otherwise they fall back to non-private names defined in the module (not just imported), plus explicitly imported sibling submodules. Curate this carefully: consumers are told to `import *` from pyskill modules, so it defines your public API.

## 2. Register via entry point

In your `pyproject.toml`:

    [project.entry-points.pyskills]
    my_skill = "mypackage.mymodule"

The key is an arbitrary name; the value is the module path.

## 3. Module contract summary

    '''Short description for discovery.

    Detailed docs read by the LLM after import.
    '''

    __all__ = ['my_func', 'MyClass']

    def my_func(x: int) -> str:
        "Does something useful"
        ...

    class MyClass:
        "A useful class"
        def method(self) -> str:
            "Does something"
            ...

After import, the LLM inspects the module with `doc(module)` (overview of classes, functions, and submodules) and `xdir(module)` (filtered list of public symbols).

## 4. Local pyskills without packaging

The entry point approach above requires a full package install. For quick personal pyskills, or pyskills shared across projects with isolated environments (e.g. separate uv venvs), pyskills provides an XDG-based pyskills directory.

When you first `import pyskills`, it creates a directory at your XDG data home (typically `~/.local/share/pyskills/`) and writes a `.pth` file into `site-packages`. This `.pth` file tells Python to add the pyskills directory to `sys.path` on startup, so any modules placed there are importable as standard Python modules without any special import machinery. This works across all Python environments on your system, even separate uv projects with isolated venvs.

You can create a pyskill programmatically with `register_pyskill`:

    from pyskills.core import register_pyskill

    register_pyskill('my_local.skill', 'A quick local pyskill.', code='''
    __all__ = ['hello']

    def hello(name: str) -> str:
        "Greet someone"
        return f"Hello, {name}!"
    ''')

This writes the module file into the XDG pyskills directory and creates a minimal dist-info entry point, so the pyskill immediately appears in `list_pyskills()`.

Use `enable_pyskill(name)` / `disable_pyskill(name)` to toggle a pyskill's visibility without deleting files. Use `pyskills_dir()` to see where the directory is.



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Key takeaways for your project, from those docs:

**The contract** — your skill module needs:
- Docstring: first paragraph = short discovery description; rest = the LLM's instruction manual
- Curated `__all__` — this *is* the public API, since consumers `import *`
- Type-annotated functions with docstrings

**Two registration paths:**

1. **Packaged** (entry point in `pyproject.toml`): `[project.entry-points.pyskills] holdem = "holdem.skill"`. This is the right final form for a deployed project.

2. **Local XDG** (`~/.local/share/pyskills/`): `register_pyskill(...)` writes the module file + dist-info entry point instantly; `list_pyskills()` sees it immediately; `enable_pyskill`/`disable_pyskill` toggle visibility. Zero packaging ceremony.

Given this is lean and you're developing inside a dialog, path 2 is ideal for iteration — you can `register_pyskill` a first version now, test discovery and loading, then move it into the package with a real entry point when you're happy.

One thing the docstring contract implies for your skill: since the LLM reads it to *learn how to teach*, the skill's module docstring should include the tutor workflow (small deck → Monte Carlo → plot) — not just function listings.

Want to draft the first version of `holdem.skill` — starting with `hand_name` and `equity` — and register it locally?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 58726, "completion_tokens": 740, "total_tokens": 59466, "cached_tokens": 57728, "reasoning_tokens": 410, "cost": 0.0017326680000000004}
```


Draft the first version for me. Here is a teaching principles by Rens, who works for answer ai:

<Teaching Principles>

You are a patient, curious, and encouraging tutor. Follow these principles:

1. **Small steps, frequent check-ins.** Present one idea at a time. After explaining or asking a question, stop and wait for the learner's response before moving on.
2. **Meet learners where they are.** Ask about their background, goals, and current understanding before diving in. Tailor your depth, vocabulary, and examples accordingly.
3. **Guide, don't solve.** Don't do the work for them. Offer hints, ask leading questions, and let them produce the answer. Only provide full solutions when explicitly asked.
4. **Build on what they know.** Connect new ideas to concepts they're already comfortable with. Use analogies from their interests when possible.
5. **Encourage active exploration.** Prefer "What do you think happens if...?" over "Here's what happens." Let them predict, try, and reflect.
6. **Embrace mistakes as learning.** Treat wrong answers as useful data. Help the learner see *why* something didn't work rather than just correcting it.
7. **Be rigorous but accessible.** Don't oversimplify to the point of being misleading. Use precise language, but define terms when they're new.
8. **Match their energy.** Mirror the learner's message length and tone. Short questions get short answers; deep questions get deeper ones. Walls of text overwhelm and disengage.
9. **Leverage the live environment.** Share small, runnable bash code snippets the learner can execute and tweak themselves. Hands-on experimentation beats passive reading.
10. **Stay curious alongside them.** Express genuine interest in the topic. Wonder out loud. Treat the learner as a collaborator, not a recipient.
11. **End responses cleanly.** Don't pad with "Let me know if...", "Feel free to ask...", or unsolicited offers. Answer, then stop. Trust the learner to drive the next step.
12. **Code samples over tool use.** To keep the learner engaged it's better to give them the codefence to run rather than you run something as a tool. Only when it is really for your own background research use tools.
</Teaching Principles>

We can proabably start with these principles as I think this is very good. Let's not focus on the teaching materials specifically. Let students' curiosity decide what they want to learn, but make sure they know prerequisite concepts. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's the draft, shaped around Rens's principles: the docstring is the LLM's teaching contract, `equity` is the Monte Carlo workhorse, `hand_name` makes showdowns explainable, and `__all__` re-exports the core API so one `import *` gives the tutor the whole toolkit.

```python
"""Teach Texas hold'em poker rules and probability interactively: Monte Carlo equity, readable hand names, and a one-import tutor toolkit.

# Teaching with this module

You are a patient, curious, encouraging tutor: small steps, guide don't solve, build on what the learner knows. Let their curiosity pick topics, but check prerequisites first — fractions/percent before pot odds, simple counting before combinations, expected value before implied odds. If a prerequisite is missing, give a two-minute primer, then return to their question.

Prefer giving the learner small runnable code cells over running tools yourself. Hands-on experimentation beats passive reading. Use tools only for your own background research.

Probability questions: use `equity` (Monte Carlo) when more than a couple of cards are unknown, and show the learner the simulation code so they can rerun it with different numbers. Start with a small deck — `mk_deck(suits[:2], ranks[:5])` — when the point is counting outcomes by hand; switch to a full deck once they're ready for real odds.

Live play: drive a hand one turn per cell — `game.start()`, then `game.act()` — so the learner can explore between turns. `always_*` strategies make bots; `human` puts a student in the seat.
"""

import random

from holdem.core import (Card, Player, Round, Game, Stage, Action,
    evaluate_hand, compare_hands, mk_deck, withdraw_card,
    always_call, always_check, always_fold, always_raise, human)

__all__ = ['Card', 'Player', 'Round', 'Game', 'Stage', 'Action',
    'evaluate_hand', 'compare_hands', 'mk_deck', 'withdraw_card',
    'always_call', 'always_check', 'always_fold', 'always_raise', 'human',
    'equity', 'hand_name']

_RANK = {2:'two',3:'three',4:'four',5:'five',6:'six',7:'seven',8:'eight',9:'nine',
         10:'ten',11:'jack',12:'queen',13:'king',14:'ace'}
_HAND = {10:'royal flush',9:'straight flush',8:'four of a kind',7:'full house',
         6:'flush',5:'straight',4:'three of a kind',3:'two pair',2:'one pair',1:'high card'}

def hand_name(rank: tuple) -> str:
    "Readable name for an `evaluate_hand` tuple, e.g. (7,3,2) → 'full house, threes over twos'."
    cat, vals = rank[0], rank[1:]
    name = _HAND[cat]
    if cat == 10: return name
    if cat in (9, 5): return f"{name}, {_RANK[vals[0]]}-high"
    if cat in (8, 4): return f"{name}, {_RANK[vals[0]]}s"
    if cat == 7: return f"{name}, {_RANK[vals[0]]}s over {_RANK[vals[1]]}s"
    if cat == 6: return f"{name}, {_RANK[vals[0]]}-high"
    if cat == 3: return f"{name}, {_RANK[vals[0]]}s and {_RANK[vals[1]]}s"
    if cat == 2: return f"{name}, {_RANK[vals[0]]}s"
    return f"{name}, {_RANK[vals[0]]}-high"

def equity(hand: list[Card], community: list[Card] = (), n_opponents: int = 1,
           n_sims: int = 10_000, seed: int = None) -> dict:
    "Monte Carlo estimate of win/tie/lose fractions for `hand` given `community`, vs `n_opponents`. Pass `seed` for reproducible classroom demos."
    if n_opponents < 1: raise ValueError('n_opponents must be >= 1')
    hand, community = list(hand), list(community)
    known = set(hand) | set(community)
    deck = [c for c in mk_deck() if c not in known]
    rng = random.Random(seed)
    wins = ties = 0
    fill = 5 - len(community)
    for _ in range(n_sims):
        rng.shuffle(deck)
        n_opp = 2*n_opponents
        board = community + deck[n_opp : n_opp+fill]
        mine = evaluate_hand(hand + board)
        best = max(evaluate_hand(deck[2*j:2*j+2] + board) for j in range(n_opponents))
        if mine > best: wins += 1
        elif mine == best: ties += 1
    return {'win': wins/n_sims, 'tie': ties/n_sims, 'lose': 1-(wins+ties)/n_sims}
```

Three design choices worth flagging:

1. **`equity` returns win/tie/lose separately** — ties are a recurring teachable moment in poker (board plays, split pots), and hiding them in a single win% would be misleading. The tutor can now say "you win 32%, tie 9%" and turn the tie into a lesson.
2. **`hand_name` covers all ten categories** with the tiebreaker ranks — `(7,3,2)` → "full house, threes over twos" — so a showdown output reads like a dealer's announcement, not a tuple.
3. **Core symbols in `__all__`** — one `from holdem.skill import *` gives the tutor everything, but core stays pure: the skill is only a curated facade plus two new helpers.

Two things to adjust on your end: the import path assumes the nbdev export lands as `holdem.core` — change it if your package layout differs. And when a learner plays, `human` is a function, not a strategy — the game flow works as you built it.

To try it, register locally and confirm discovery:

```python
from pyskills.core import register_pyskill
register_pyskill('holdem.skill', 'Teach hold\'em rules and probability interactively.', code=open('holdem/skill.py').read())
```

then `list_pyskills()` should show it.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 59462, "completion_tokens": 9492, "total_tokens": 68954, "cached_tokens": 58880, "reasoning_tokens": 7996, "cost": 0.0053453400000000005}
```


Actually I think `hand_name` and `equity` do not have to be inside of the pyskill file. I think it is better if they are inside of the `core.py`.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Makes sense — they're generally useful poker utilities, not teaching-specific: the GUI will want `hand_name` at showdown, and an AI bot could use `equity` to decide. Core stays the single home for game logic; the skill just re-exports.

The concrete plan, pending your ok:

1. **In `00_core.ipynb`**, add two `#| export` cells (e.g. after `compare_hands`): `hand_name` and `equity`, as drafted — they only depend on core symbols (`mk_deck`, `evaluate_hand`, `rank_values`).
2. **In the skill module**, delete those two definitions; keep them in `__all__` but import them from core (add to the existing core import line) so the tutor's one-import experience is unchanged.
3. Re-export core → the skill still compiles, `equity`/`hand_name` now live in `core.py`.

One naming thought while we're at it: with the helpers in core, `holdem.skill`'s own `__all__` is just a curated view of core — which is exactly the facade pattern we agreed on.

Want me to draft the two notebook cells ready for `#| export`, or will you place them yourself?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 61055, "completion_tokens": 913, "total_tokens": 61968, "cached_tokens": 59392, "reasoning_tokens": 639, "cost": 0.0019799220000000003}
```


Add two notebook cells for me.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Adding the two exported cells after `test_compare_hands`, before the Deck section. First cell — the rank/hand-name maps plus `hand_name`:
- ⏳ `py(code="cell1 = r'''#| export\n_RANK = {2:'two',3…")` ⏳


```json {.tool}
{
  "id": "call_00_8f7jAQG2kGlFweDmHLNy9577",
  "name": "py",
  "args": {
    "code": "cell1 = r'''#| export\n_RANK = {2:'two',3:'three',4:'four',5:'five',6:'six',7:'seven',8:'eight',9:'nine',\n         10:'ten',11:'jack',12:'queen',13:'king',14:'ace'}\n_HAND = {10:'royal flush',9:'straight flush',8:'four of a kind',7:'full house',\n         6:'flush',5:'straight',4:'three of a kind',3:'two pair',2:'one pair',1:'high card'}\n\ndef hand_name(rank: tuple) -> str:\n    \"Readable name for an `evaluate_hand` tuple, e.g. (7,3,2) -> 'full house, threes over twos'.\"\n    cat, vals = rank[0], rank[1:]\n    name = _HAND[cat]\n    if cat == 10: return name\n    if cat in (9, 5): return f\"{name}, {_RANK[vals[0]]}-high\"\n    if cat in (8, 4): return f\"{name}, {_RANK[vals[0]]}s\"\n    if cat == 7: return f\"{name}, {_RANK[vals[0]]}s over {_RANK[vals[1]]}s\"\n    if cat == 6: return f\"{name}, {_RANK[vals[0]]}-high\"\n    if cat == 3: return f\"{name}, {_RANK[vals[0]]}s and {_RANK[vals[1]]}s\"\n    if cat == 2: return f\"{name}, {_RANK[vals[0]]}s\"\n    return f\"{name}, {_RANK[vals[0]]}-high\"\n'''\ncid1 = await add_msg(cell1, msg_type='code', placement='add_after', id='a05fffc5')\ncid1"
  },
  "result": "491fa4c1"
}
```





First cell added (#_491fa4c1). Now the `equity` cell, placed right after it:
- ⏳ `py(code="cell2 = r'''#| export\ndef equity(hand: l…")` ⏳


```json {.tool}
{
  "id": "call_00_BgAsEm94g1vtC52owaD41485",
  "name": "py",
  "args": {
    "code": "cell2 = r'''#| export\ndef equity(hand: list[Card], community: list[Card] = (), n_opponents: int = 1,\n           n_sims: int = 10_000, seed: int = None) -> dict:\n    \"Monte Carlo estimate of win/tie/lose fractions for `hand` given `community`, vs `n_opponents`. Pass `seed` for reproducible classroom demos.\"\n    if n_opponents < 1: raise ValueError('n_opponents must be >= 1')\n    hand, community = list(hand), list(community)\n    known = set(hand) | set(community)\n    deck = [c for c in mk_deck() if c not in known]\n    rng = random.Random(seed)\n    wins = ties = 0\n    fill = 5 - len(community)\n    for _ in range(n_sims):\n        rng.shuffle(deck)\n        n_opp = 2*n_opponents\n        board = community + deck[n_opp : n_opp+fill]\n        mine = evaluate_hand(hand + board)\n        best = max(evaluate_hand(deck[2*j:2*j+2] + board) for j in range(n_opponents))\n        if mine > best: wins += 1\n        elif mine == best: ties += 1\n    return {'win': wins/n_sims, 'tie': ties/n_sims, 'lose': 1-(wins+ties)/n_sims}\n'''\ncid2 = await add_msg(cell2, msg_type='code', placement='add_after', id=cid1)\ncid2"
  },
  "result": "bc0bcf43"
}
```

Both cells added, in order:

- **#_491fa4c1** — `_RANK`/`_HAND` maps + `hand_name`
- **#_bc0bcf43** — `equity`

Both are marked `#| export` so they'll land in `core.py` on the next nbdev export. They're added unrun — run them when you want to test. A couple of things worth checking once they're in:

1. **`equity` edge cases**: pre-flop (`community=[]`), and a nearly-complete board (`community` length 4) both work — `fill` handles 0 to 5. Worth a quick test of each, plus `n_opponents` and `seed` for reproducibility.
2. **`hand_name` coverage**: run it across all ten categories — the royal flush branch (`(10,)` → `"royal flush"`) is the only one with no extra ranks to read.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 193658, "completion_tokens": 5151, "total_tokens": 198809, "cached_tokens": 193024, "reasoning_tokens": 4001, "cost": 0.006350064000000001}
```


ok I added tests for the `hand_name` and played around with `equity`, which was very fun. I think the students would love it.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice — tests passing and `equity` being fun is a good sign: that's the same loop students will hit ("what if I run it with 2 opponents? with a seed?"). The Monte Carlo exploration you did *is* the lesson.

Two next-step directions, pick whichever:

1. **Finish the skill** — now that `hand_name`/`equity` live in core, the skill module becomes a pure facade: teaching docstring + curated `__all__` re-exporting core. We could draft that now.

2. **Try the teaching loop** — draft a sample lesson (say, pot odds or "why do aces lose to more opponents?") as the pyskill's docstring, so you can feel how the LLM would use it.

Which one?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 66440, "completion_tokens": 168, "total_tokens": 66608, "cached_tokens": 49024, "cost": 0.0047574240000000005}
```


Let's finish the skill. The goal is to teach students and let the explore. Ask students what action they would like to perform and why? It's okay to just play the game for fun: no need to understand all the details and concepts related to poker. Wait until they want to dig deeper into some fun concepts they are excited to learn more about. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check the project layout so the skill lands in the right place:
- ⏳ `py(code="ls('../..', files=False, dirs=True), ls(…")` ⏳


```json {.tool}
{
  "id": "call_00_NUGbvKGgn92tRHjCHVBW3839",
  "name": "py",
  "args": {
    "code": "ls('../..', files=False, dirs=True), ls('..', files=False, dirs=True)"
  },
  "result": "(drwxrwsr-x    4.0K 2026-08-08 17:15 holdem\ndrwxrwsr-x    4.0K 2026-07-22 22:46 hypermedia\ndrwxrwsr-x    4.0K 2026-08-06 03:04 novel,\n drwxrwsr-x    4.0K 2026-08-08 17:15 _proc\ndrwxrwsr-x    4.0K 2026-08-13 20:48 holdem\ndrwxrwsr-x    4.0K 2026-06-28 18:51 holdem.egg-info\ndrwxrwsr-x    4.0K 2026-08-13 22:09 nbs)"
}
```




- ⏳ `py(code="ls('../holdem')")` ⏳


```json {.tool}
{
  "id": "call_00_z47aezWG05P0H4G9YvOf3962",
  "name": "py",
  "args": {
    "code": "ls('../holdem')"
  },
  "result": "-rw-rw-r--      42 2026-08-13 20:48 __init__.py\ndrwxrwsr-x    4.0K 2026-08-08 17:15 __pycache__\n-rw-rw-r--    5.5K 2026-08-13 20:48 _modidx.py\n-rw-rw-r--   12.5K 2026-08-13 20:48 core.py"
}
```





Package root is `../holdem/` (with `core.py` already exported there), so the skill becomes `holdem/skill.py`. Here's the finished module — pure facade, with the teaching contract front and center:

```python
"""Teach Texas hold'em by playing: let learners drive the game, ask what they'd do and why, and dive into concepts only when curiosity strikes.

# Teaching approach

Your job is to get the learner playing and having fun — no lectures up front. Open with an invitation to play, and let them choose their own adventure: rules, strategy, probability, or just play for fun. It's fine to play hands without understanding every detail.

When a learner says what action they'd like to take, ask them *why* — their reasoning is the teaching moment, whether right or wrong. Treat wrong answers as useful data, and help them see why something didn't work rather than just correcting it.

Do not front-load concepts. Play first; explain only what they ask about. When curiosity lands on something (pot odds, equity, variance...), check prerequisites before diving in: fractions/percent before pot odds, simple counting before combinations, expected value before implied odds. If a prerequisite is missing, give a two-minute primer, then return to their question. Let the learner's curiosity decide what to learn next.

Work in small steps: one idea at a time, then stop and wait. Match the learner's energy — short questions get short answers. Be rigorous but accessible: precise language, terms defined when new.

# Using the tools

Drive live play one turn per cell, so the learner can explore between turns:

    from holdem.skill import *
    game = Game([Player('you', balance=20, strategy=human),
                 Player('bot', balance=20, strategy=always_call)])
    game.start()   # deal and show your hand
    game.act()     # one human turn; bots auto-play; repeat until "Game is over"

`always_*` strategies make bots with different personalities; `human` puts a learner in the seat. `game.start()` again deals a fresh hand with the same players.

For probability questions use `equity` (Monte Carlo) when more than a couple of cards are unknown, and show the learner the code so they can rerun it with different numbers. When a small deck makes the point clearer (counting outcomes by hand), build one with `mk_deck(suits[:2], ranks[:5])` and switch to the full deck once they're ready for real odds.

Prefer giving the learner small runnable code cells over running tools yourself — hands-on experimentation beats passive reading. Use tools only for your own background research.
"""

from holdem.core import (Card, Player, Round, Game, Stage, Action,
    evaluate_hand, compare_hands, hand_name, equity, mk_deck, withdraw_card,
    always_call, always_check, always_fold, always_raise, human)

__all__ = ['Card', 'Player', 'Round', 'Game', 'Stage', 'Action',
    'evaluate_hand', 'compare_hands', 'hand_name', 'equity', 'mk_deck',
    'withdraw_card', 'always_call', 'always_check', 'always_fold',
    'always_raise', 'human']
```

Two notes:

1. **The docstring is the whole product** — the LLM reads it to learn how to teach, so the opening line carries your philosophy: "let learners drive, ask what they'd do and why, dive in only when curiosity strikes." The rest is workflow so the LLM knows *how* to run a hand and when to reach for `equity` vs a small deck.
2. **No new logic** — `hand_name`/`equity` now come from core; the skill is just the curated API plus the teaching manual. `Game`/`Player`/`Stage`/etc. re-exported so one `import *` is enough.

Want me to write it to `../holdem/skill.py`, or will you place it yourself?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 204423, "completion_tokens": 3027, "total_tokens": 207450, "cached_tokens": 203904, "reasoning_tokens": 2040, "cost": 0.005662314000000001}
```


Write it to the file plz.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Writing the skill module to `../holdem/skill.py`:
- ⏳ `py(code="skill_src = r'''\"\"\"Teach Texas hold'em b…")` ⏳


```json {.tool}
{
  "id": "call_00_44nyQIon7aHhovny800Q8665",
  "name": "py",
  "args": {
    "code": "skill_src = r'''\"\"\"Teach Texas hold'em by playing: let learners drive the game, ask what they'd do and why, and dive into concepts only when curiosity strikes.\n\n# Teaching approach\n\nYour job is to get the learner playing and having fun — no lectures up front. Open with an invitation to play, and let them choose their own adventure: rules, strategy, probability, or just play for fun. It's fine to play hands without understanding every detail.\n\nWhen a learner says what action they'd like to take, ask them *why* — their reasoning is the teaching moment, whether right or wrong. Treat wrong answers as useful data, and help them see why something didn't work rather than just correcting it.\n\nDo not front-load concepts. Play first; explain only what they ask about. When curiosity lands on something (pot odds, equity, variance...), check prerequisites before diving in: fractions/percent before pot odds, simple counting before combinations, expected value before implied odds. If a prerequisite is missing, give a two-minute primer, then return to their question. Let the learner's curiosity decide what to learn next.\n\nWork in small steps: one idea at a time, then stop and wait. Match the learner's energy — short questions get short answers. Be rigorous but accessible: precise language, terms defined when new.\n\n# Using the tools\n\nDrive live play one turn per cell, so the learner can explore between turns:\n\n    from holdem.skill import *\n    game = Game([Player('you', balance=20, strategy=human),\n                 Player('bot', balance=20, strategy=always_call)])\n    game.start()   # deal and show your hand\n    game.act()     # one human turn; bots auto-play; repeat until \"Game is over\"\n\n`always_*` strategies make bots with different personalities; `human` puts a learner in the seat. `game.start()` again deals a fresh hand with the same players.\n\nFor probability questions use `equity` (Monte Carlo) when more than a couple of cards are unknown, and show the learner the code so they can rerun it with different numbers. When a small deck makes the point clearer (counting outcomes by hand), build one with `mk_deck(suits[:2], ranks[:5])` and switch to the full deck once they're ready for real odds.\n\nPrefer giving the learner small runnable code cells over running tools yourself — hands-on experimentation beats passive reading. Use tools only for your own background research.\n\"\"\"\n\nfrom holdem.core import (Card, Player, Round, Game, Stage, Action,\n    evaluate_hand, compare_hands, hand_name, equity, mk_deck, withdraw_card,\n    always_call, always_check, always_fold, always_raise, human)\n\n__all__ = ['Card', 'Player', 'Round', 'Game', 'Stage', 'Action',\n    'evaluate_hand', 'compare_hands', 'hand_name', 'equity', 'mk_deck',\n    'withdraw_card', 'always_call', 'always_check', 'always_fold',\n    'always_raise', 'human']\n'''\ncreate_file('../holdem/skill.py', skill_src)"
  },
  "result": "PermissionError: open '../holdem/skill.py' not in (). Code message has been added to dialog. Ask user to run it."
}
```





The file write was blocked. `create_file('../holdem/skill.py', ...)` raised `PermissionError: open '../holdem/skill.py' not in ()` — the sandbox's audit system doesn't permit writes outside the current working directory, so the relative path `../holdem/skill.py` isn't accessible. The full skill module source was added to the dialog as a code message for you to run manually.

Two ways forward:

1. **Run the auto-added code cell** — it contains `create_file('../holdem/skill.py', skill_src)`; running it from your side will create the file.
2. **Tell me the correct path** — if there's an allowed location (e.g. a `holdem/` folder inside the current working directory), I can write it there instead.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 135654, "completion_tokens": 1772, "total_tokens": 137426, "cached_tokens": 103296, "reasoning_tokens": 882, "cost": 0.009708636}
```


In [ ]:
skill_src = r'''"""Teach Texas hold'em by playing: let learners drive the game, ask what they'd do and why, and dive into concepts only when curiosity strikes.

# Teaching approach

Your job is to get the learner playing and having fun — no lectures up front. Open with an invitation to play, and let them choose their own adventure: rules, strategy, probability, or just play for fun. It's fine to play hands without understanding every detail.

When a learner says what action they'd like to take, ask them *why* — their reasoning is the teaching moment, whether right or wrong. Treat wrong answers as useful data, and help them see why something didn't work rather than just correcting it.

Do not front-load concepts. Play first; explain only what they ask about. When curiosity lands on something (pot odds, equity, variance...), check prerequisites before diving in: fractions/percent before pot odds, simple counting before combinations, expected value before implied odds. If a prerequisite is missing, give a two-minute primer, then return to their question. Let the learner's curiosity decide what to learn next.

Work in small steps: one idea at a time, then stop and wait. Match the learner's energy — short questions get short answers. Be rigorous but accessible: precise language, terms defined when new.

# Using the tools

Drive live play one turn per cell, so the learner can explore between turns:

    from holdem.skill import *
    game = Game([Player('you', balance=20, strategy=human),
                 Player('bot', balance=20, strategy=always_call)])
    game.start()   # deal and show your hand
    game.act()     # one human turn; bots auto-play; repeat until "Game is over"

`always_*` strategies make bots with different personalities; `human` puts a learner in the seat. `game.start()` again deals a fresh hand with the same players.

For probability questions use `equity` (Monte Carlo) when more than a couple of cards are unknown, and show the learner the code so they can rerun it with different numbers. When a small deck makes the point clearer (counting outcomes by hand), build one with `mk_deck(suits[:2], ranks[:5])` and switch to the full deck once they're ready for real odds.

Prefer giving the learner small runnable code cells over running tools yourself — hands-on experimentation beats passive reading. Use tools only for your own background research.
"""

from holdem.core import (Card, Player, Round, Game, Stage, Action,
    evaluate_hand, compare_hands, hand_name, equity, mk_deck, withdraw_card,
    always_call, always_check, always_fold, always_raise, human)

__all__ = ['Card', 'Player', 'Round', 'Game', 'Stage', 'Action',
    'evaluate_hand', 'compare_hands', 'hand_name', 'equity', 'mk_deck',
    'withdraw_card', 'always_call', 'always_check', 'always_fold',
    'always_raise', 'human']
'''
create_file('../holdem/skill.py', skill_src, overwrite=True)

'done'

In [ ]:
list_pyskills()

{'fastmux.skill': 'Live handles for tmux sessions, windows, and panes, plus named background sessions driven by sid. Use this when code needs to read terminal screens or scrollback, drive interactive processes, build pane layouts, search text across terminals, or keep a persistent terminal session that both an agent and the user can inspect and drive.',
 'rgapi.skill': 'Fast and flexible file discovery and search for Python. Use this when code needs `fd`-style file finding or `rg`-style searching.',
 'holdem.skill': "Teach hold'em rules and probability interactively.",
 'llmsurgery.skill': 'Read and work with Claude Code and Codex session transcripts.',
 'exhash.skill': 'Universal hash-verified text editing for local files. Use this when an LLM needs one safe editing interface for reading, previewing, and modifying text files.',
 'dialoghelper.solveitskill': 'Read, search, edit, and manage Solveit dialogs using dialoghelper.core, including dialog/message addressing, line-numbered inspe

In [ ]:
from pyskills.core import register_pyskill

how do I separate the docstr and code from the file?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Parse the file with `ast` — it finds the docstring boundary reliably, and everything after it is the code:

```python
import ast
src = Path('../holdem/skill.py').read_text()
tree = ast.parse(src)
docstr = ast.get_docstring(tree)
code = '\n'.join(src.splitlines()[tree.body[0].end_lineno:])
```

Then register with the two pieces:

```python
register_pyskill('holdem.skill', docstr, code)
```

`register_pyskill` rebuilds the module as `"""docstr"""` + code in the XDG pyskills dir, and the first line of `docstr` becomes the discovery description in `list_pyskills()` — so this also upgrades the description from "Teach hold'em rules and probability interactively." to the fuller "Teach Texas hold'em by playing…".

One caveat: `register_pyskill` creates a *standalone* `holdem/skill.py` under the XDG dir with its own empty `holdem/__init__.py`, so `from holdem.core import ...` inside it depends on the real `holdem` package winning the import — a potential shadowing gotcha. If it fails to import, the cleaner alternative is the packaged route: add `holdem = "holdem.skill"` under `[project.entry-points.pyskills]` in your `pyproject.toml` and reinstall editable.

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 70381, "completion_tokens": 3488, "total_tokens": 73869, "cached_tokens": 5376, "reasoning_tokens": 3175, "cost": 0.015228906}
```


In [ ]:
import ast
src = Path('../holdem/skill.py').read_text()
src

'"""Teach Texas hold\'em by playing: let learners drive the game, ask what they\'d do and why, and dive into concepts only when curiosity strikes.\n\n# Teaching approach\n\nYour job is to get the learner playing and having fun — no lectures up front. Open with an invitation to play, and let them choose their own adventure: rules, strategy, probability, or just play for fun. It\'s fine to play hands without understanding every detail.\n\nWhen a learner says what action they\'d like to take, ask them *why* — their reasoning is the teaching moment, whether right or wrong. Treat wrong answers as useful data, and help them see why something didn\'t work rather than just correcting it.\n\nDo not front-load concepts. Play first; explain only what they ask about. When curiosity lands on something (pot odds, equity, variance...), check prerequisites before diving in: fractions/percent before pot odds, simple counting before combinations, expected value before implied odds. If a prerequisite is 

In [ ]:
tree = ast.parse(src)
tree

In [ ]:
docstr = ast.get_docstring(tree)
docstr

'Teach Texas hold\'em by playing: let learners drive the game, ask what they\'d do and why, and dive into concepts only when curiosity strikes.\n\n# Teaching approach\n\nYour job is to get the learner playing and having fun — no lectures up front. Open with an invitation to play, and let them choose their own adventure: rules, strategy, probability, or just play for fun. It\'s fine to play hands without understanding every detail.\n\nWhen a learner says what action they\'d like to take, ask them *why* — their reasoning is the teaching moment, whether right or wrong. Treat wrong answers as useful data, and help them see why something didn\'t work rather than just correcting it.\n\nDo not front-load concepts. Play first; explain only what they ask about. When curiosity lands on something (pot odds, equity, variance...), check prerequisites before diving in: fractions/percent before pot odds, simple counting before combinations, expected value before implied odds. If a prerequisite is mis

In [ ]:
tree.body[0].end_lineno

28

In [ ]:
code = '\n'.join(src.splitlines()[tree.body[0].end_lineno:])
code

"\nfrom holdem.core import (Card, Player, Round, Game, Stage, Action,\n    evaluate_hand, compare_hands, hand_name, equity, mk_deck, withdraw_card,\n    always_call, always_check, always_fold, always_raise, human)\n\n__all__ = ['Card', 'Player', 'Round', 'Game', 'Stage', 'Action',\n    'evaluate_hand', 'compare_hands', 'hand_name', 'equity', 'mk_deck',\n    'withdraw_card', 'always_call', 'always_check', 'always_fold',\n    'always_raise', 'human']"

In [ ]:
register_pyskill('holdem.skill', docstr, code)

In [ ]:
list_pyskills()

{'fastmux.skill': 'Live handles for tmux sessions, windows, and panes, plus named background sessions driven by sid. Use this when code needs to read terminal screens or scrollback, drive interactive processes, build pane layouts, search text across terminals, or keep a persistent terminal session that both an agent and the user can inspect and drive.',
 'rgapi.skill': 'Fast and flexible file discovery and search for Python. Use this when code needs `fd`-style file finding or `rg`-style searching.',
 'holdem.skill': "Teach Texas hold'em by playing: let learners drive the game, ask what they'd do and why, and dive into concepts only when curiosity strikes.",
 'llmsurgery.skill': 'Read and work with Claude Code and Codex session transcripts.',
 'exhash.skill': 'Universal hash-verified text editing for local files. Use this when an LLM needs one safe editing interface for reading, previewing, and modifying text files.',
 'dialoghelper.solveitskill': 'Read, search, edit, and manage Solveit

Got the skill! But I don't like the description of the skill. I feel like it should at least mention Texas Hold'em poker. It might be worth adding other details..

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check `register_pyskill`'s docs to see how to update an existing registration:
- ⏳ `py(code="from pyskills.core import register_pyski…")` ⏳


````json {.tool}
{
  "id": "call_00_JDiSGpZeHMxBaaLiXMys0525",
  "name": "py",
  "args": {
    "code": "from pyskills.core import register_pyskill\ndoc(register_pyskill)"
  },
  "result": "---\n\n[source](https://github.com/AnswerDotAI/pyskills/blob/main/pyskills/core.py#LNone){target=\"_blank\" style=\"float:right; font-size:smaller\"}\n\n### register_pyskill\n\n```python\ndef register_pyskill(\n    name, docstr, code:str=''\n):\n```\n\n*Register a pyskill module `name` in the xdg pyskills dir*"
}
````





The real signature is `register_pyskill(name, docstr, code='')` — so the second argument is the description that shows in `list_pyskills()`, stored at registration time. Re-registering with the same name and a new `docstr` should update it. Let me quickly peek at what was written, to confirm:
- ⏳ `py(code="from pyskills.core import pyskills_dir\ni…")` ⏳


```json {.tool}
{
  "id": "call_00_4U01L1M8EFH7G31FsE177050",
  "name": "py",
  "args": {
    "code": "from pyskills.core import pyskills_dir\nimport glob\npyskills_dir(), glob.glob(pyskills_dir()+'/**', recursive=True)[:20]"
  },
  "result": "---------------------------------------------------------------------------\nTypeError                                 Traceback (most recent call last)\nCell In[121], line 1\n----> 1 await call_tool(py, {'code': \"from pyskills.core import pyskills_dir\\nimport glob\\npyskills_dir(), glob.glob(pyskills_dir()+'/**', recursive=True)[:20]\"})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:351, in call_tool(func, kw)\n    349 async def call_tool(func, kw):\n    350     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 351     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/aio.py:88, in maybe_await(o)\n     86 \"Await `o` if needed, and return it\"\n     87 from inspect import isawaitable\n---> 88 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:341, in RunPython.__call__(self, code)\n    339 tb = e.__traceback__\n    340 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 341 raise e.with_traceback(tb) from None\n\nFile <python_4>:1\n----> 1 (pyskills_dir(), glob.glob(pyskills_dir() + '/**', recursive=True)[:20])\n\nTypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'"
}
```

Minor error — `pyskills_dir()` returns a `PosixPath`, not a string. Let me retry with proper path joining:
- ⏳ `py(code="from pyskills.core import pyskills_dir\nd…")` ⏳


```json {.tool}
{
  "id": "call_00_ET_h4u8a2vU9q1GhUgtRyYT0570",
  "name": "py",
  "args": {
    "code": "from pyskills.core import pyskills_dir\nd = pyskills_dir()\nd, [str(p.relative_to(d)) for p in d.rglob('*')][:30]"
  },
  "result": "(Path('/app/data/.local/share/pyskills'),\n ['holdem_skill-0.0.1.dist-info',\n  'holdem',\n  'holdem_skill-0.0.1.dist-info/entry_points.txt',\n  'holdem_skill-0.0.1.dist-info/METADATA',\n  'holdem/__init__.py',\n  'holdem/__pycache__',\n  'holdem/skill.py',\n  'holdem/__pycache__/__init__.cpython-312.pyc'])"
}
```


- ⏳ `py(code="print(Path(d/'holdem_skill-0.0.1.dist-in…")` ⏳


```json {.tool}
{
  "id": "call_00_ET_YOSr0unvk3BDNFvfIggG1434",
  "name": "py",
  "args": {
    "code": "print(Path(d/'holdem_skill-0.0.1.dist-info'/'entry_points.txt').read_text())\nprint('---')\nprint(Path(d/'holdem_skill-0.0.1.dist-info'/'METADATA').read_text()[:300])"
  },
  "result": "[pyskills]\nholdem.skill = holdem.skill\n\n---\nName: holdem_skill\nVersion: 0.0.1"
}
```





The entry point looks right. Now let me see how the description is stored — reading the registered copy of the skill module:
- ⏳ `py(code="print(Path(d/'holdem'/'skill.py').read_t…")` ⏳


```json {.tool}
{
  "id": "call_00_N1ngPA6fdrUvwCtmNJtz3977",
  "name": "py",
  "args": {
    "code": "print(Path(d/'holdem'/'skill.py').read_text()[:600])"
  },
  "result": "\"\"\"Teach hold'em rules and probability interactively.\"\"\"\n\"\"\"Teach Texas hold'em by playing: let learners drive the game, ask what they'd do and why, and dive into concepts only when curiosity strikes.\n\n# Teaching approach\n\nYour job is to get the learner playing and having fun — no lectures up front. Open with an invitation to play, and let them choose their own adventure: rules, strategy, probability, or just play for fun. It's fine to play hands without understanding every detail.\n\nWhen a learner says what action they'd like to take, ask them *why* — their reasoning is the teaching moment, wh"
}
```





Found something important while checking the storage: `register_pyskill` writes the module as your `docstr` string, followed by your `code` — it *prepends* the docstr. The registered `skill.py` now looks like:

```python
"""Teach hold'em rules and probability interactively."""
"""Teach Texas hold'em by playing: let learners drive the game, ...
```

Two consequences:

1. **`__doc__` is only the first short string** — the teaching manual is a dangling second string, so an LLM loading the skill via `doc()` would see *only* the one-liner, not the whole teaching approach. The manual would be silently lost.
2. **`list_pyskills()` shows the first line of the module docstring** — so the description you pass to `register_pyskill` is what discovery sees.

The clean fix: re-register with `docstr` = the *full* teaching docstring (first line = discovery description, rest = the manual), and `code` = `skill.py` minus its leading docstring. Then discovery shows the first line, and `doc()` shows the whole manual.

Proposed new discovery line — mentions Texas Hold'em and what it offers:

> Interactive Texas Hold'em poker tutor: play hands against bots and learn rules and probability through Monte Carlo equity simulations.

So the re-registration would be:

```python
from pyskills.core import register_pyskill
import re
src = open('../holdem/skill.py').read()
docstr, code = re.match(r'"""(.*?)"""\s*(.*)', src, re.S).groups()
register_pyskill('holdem.skill', docstr, code=code)
```

with the new first line swapped into `docstr`. Want me to apply that?

```json {.usage}
{"model": "deepseek-v4-flash", "prompt_tokens": 442923, "completion_tokens": 5109, "total_tokens": 448032, "cached_tokens": 441728, "reasoning_tokens": 4199, "cost": 0.011673018}
```


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()